# Harvest of the test set

This notebook builds the test set of real mutated Australian addresses. It harvests addresses from court decisions in the Open Australian Legal Corpus, matches each one back to G-NAF to find the correct address, and labels the kinds of mistake each one contains. It is the first notebook in the pipeline.

- Reads: the Open Australian Legal Corpus (`corpus.jsonl`) and the G-NAF Core and authority code files, from the folder set by `HARVEST_BASE` (cell 3).
- Writes: `gnaf.duckdb` (the G-NAF tables used by every later notebook), `harvest/`, `cache/` and `output/`. The main output is `output/misspellings.tsv`, the test set before the manual label review. `output/README.md` lists every output file.
- Run after: first.

The first run is long. Most of the time goes on the corpus harvest (cell 8), the G-NAF ingest (cell 9) and the exhaustive match (cell 12). Each stage caches its result, so later runs take minutes. The `FORCE` set in cell 4 rebuilds a chosen stage.

In [ ]:
# Cell 2: install the packages the notebook imports.

%pip install --upgrade pip
%pip install duckdb orjson tqdm pyarrow pandas rapidfuzz

In [ ]:
# Cell 3: every file and folder the notebook reads or writes, declared once.
# Later cells use these names and build no paths of their own.

import os
from pathlib import Path

# Folder that holds the downloaded G-NAF and corpus files.
# Set HARVEST_BASE to use a different folder.
BASE = Path(os.environ.get("HARVEST_BASE", "data"))

# External source data. The notebook cannot rebuild these.
AUT    = BASE / "GNAF/GNAF/g-naf_may26_allstates_gda2020_psv_1023/G-NAF/G-NAF MAY 2026/Authority Code"
CORE   = BASE / "GNAF/GNAF Core/G-NAF Core/G-NAF Core MAY 2026/Standard"
CORPUS = BASE / "Open Australian Legal Corpus/corpus.jsonl"

# the field inside each corpus JSON line that holds the document text
TEXT_FIELD = "text"

# Derived folders. The notebook rebuilds them if they are deleted.
HARVEST = Path("harvest")    # corpus harvest results (slow to rebuild)
CACHE   = Path("cache")      # intermediate caches
EXPORT  = Path("output")     # final outputs
DB_PATH = Path("gnaf.duckdb")# G-NAF tables and the exhaustive match
OUT = EXPORT

# harvest/, written by cell 8 and read by cell 10
HARVEST_PARTS    = "part-*.parquet"            # the harvested addresses, in chunks
HARVEST_PROGRESS = HARVEST / "_progress.json"  # resume state. Delete it only together
                                               # with the part files.

# cache/, each file written by one cell and re-read by it on later runs
DIAG_PQ     = CACHE / "diag.parquet"      # cell 10: normalised and diagnosed unmatched set
TO_MATCH_PQ = CACHE / "to_match.parquet"  # cell 11: cleaned set for exhaustive matching
RESOLVED_PQ = CACHE / "resolved.parquet"  # cell 14: candidates with numbers attached

# output/, the products, each described in output/README.md (cell 26)
ALL_CANDIDATES_PQ = EXPORT / "resolved_all_candidates.parquet"  # cell 14: top 25 candidates per address
COLLOQUIAL_CSV    = EXPORT / "colloquial_locality_map.csv"      # cell 17: colloquial suburb -> gazetted suburb lookup
CLASSIFIED_PQ     = EXPORT / "resolved_classified.parquet"      # cell 19: RESOLVED rows with final classes and error field
CHAR_OPS_PQ       = EXPORT / "char_edit_ops.parquet"            # cell 20: character-level edit operations
MISSPELLINGS_TSV  = EXPORT / "misspellings.tsv"                 # cell 21: address / matched / reason, RETIRED excluded
MUT_TYPES_TSV     = EXPORT / "mutation_types.tsv"               # cell 22: mutation-type summary with percentages
MUT_LONG_TSV      = EXPORT / "mutations_long.tsv"               # cell 22: one row per (address, mutation)
RESOLVED_BEST_PQ  = EXPORT / "resolved_best.parquet"            # cell 23: one best match per address, fully classified
MISTAKES_TSV      = EXPORT / "mistake_types.tsv"                # cell 24: mistake counts on strict and wide bases
SUBSTITUTIONS_TSV = EXPORT / "substitutions.tsv"                # cell 25: character substitutions with categories
README_MD         = EXPORT / "README.md"                        # cell 26: describes every file in output/

print(f"BASE    {BASE}")
print(f"AUT     {AUT}")
print(f"CORE    {CORE}")
print(f"CORPUS  {CORPUS}")

In [ ]:
# Cell 4: preflight checks and run controls. Stops at once if an external
# source file is missing, or if the harvest progress file exists without its
# part files (cell 8 would then wrongly treat the harvest as finished).

# Stages to rebuild on this run. Add a key and re-run from that cell onwards.
#   "gnaf"       cell  9: G-NAF ingest into gnaf.duckdb
#   "diag"       cell 10: normalise and diagnose the harvest
#   "clean"      cell 11: strip redactions and bleed
#   "exhaustive" cell 12: exhaustive match (drops street_matches and restarts)
#   "resolve"    cell 14: attach house numbers to candidates
# Example: FORCE = {"diag", "clean"}
# To start again from nothing, delete harvest/, cache/, output/ and
# gnaf.duckdb, then restart the kernel.

FORCE = set()

# External sources must exist before anything runs.
AUT_FILES = [AUT / f"Authority_Code_{n}_psv.psv"
             for n in ("STREET_TYPE_AUT", "STREET_SUFFIX_AUT",
                       "FLAT_TYPE_AUT", "LEVEL_TYPE_AUT")]

# If the CORE folder holds no .psv files, check the expected file name so the
# report below lists it as MISSING.
core_files = sorted(CORE.glob("*.psv"))
if not core_files:
    core_files = [CORE / "GNAF_CORE.psv"]

problems = []
print("external source data:")
for p in [CORPUS, *core_files, *AUT_FILES]:
    if p.exists():
        print(f"  ok      {p.stat().st_size/1e9:7.2f} GB  {p.name}")
    else:
        problems.append(p)
        print(f"  MISSING                {p}")

if problems:
    raise FileNotFoundError(
        "External source data is missing (listed above as MISSING). "
        "These files cannot be regenerated by this notebook. "
        "Fix the paths in cell 3 or restore the files, then re-run from cell 3.")

# Harvest progress file without its part files.
if HARVEST_PROGRESS.exists() and not list(HARVEST.glob(HARVEST_PARTS)):
    raise RuntimeError(
        "harvest/_progress.json exists but there are no part-*.parquet files. "
        "Cell 8 would wrongly believe the harvest is complete and write nothing. "
        "Delete harvest/_progress.json (or the whole harvest/ folder) and re-run.")

print("\npreflight passed.")

In [ ]:
# Cell 5: imports, the database connection and the helpers used throughout.
# _s() turns any value into a string, with "" for a missing value. NaN is
# truthy in Python, so every value that may be missing goes through _s()
# before it is tested or handled as a string.

import re, json as stdjson, os, time
import duckdb
import pandas as pd
import numpy as np

try:
    import orjson as _json
    loads = _json.loads          # orjson parses the 9.4 GB corpus faster
except ImportError:
    loads = stdjson.loads

for folder in (HARVEST, CACHE, EXPORT):
    folder.mkdir(exist_ok=True)

con = duckdb.connect(str(DB_PATH))

def _s(x):
    """Anything -> str. NaN is truthy, so always coerce before testing."""
    return "" if x is None or (isinstance(x, float) and x != x) or x is pd.NA else str(x)

def table_exists(name):
    return con.execute(
        "SELECT count(*) FROM duckdb_tables() WHERE table_name = ?", [name]
    ).fetchone()[0] > 0

def table_rows(name):
    return (con.execute(f'SELECT count(*) FROM "{name}"').fetchone()[0]
            if table_exists(name) else 0)

def cached(name, build, force=False):
    """Load cache/<name>.parquet if present, else run build() and save it.
       Pass force=True (via the FORCE set in cell 4) to rebuild regardless."""
    path = CACHE / f"{name}.parquet"
    if path.exists() and not force:
        df = pd.read_parquet(path)
        print(f"[cache] loaded {name}: {len(df):,} rows")
        return df
    print(f"[build] {name} ...")
    df = build()
    df.to_parquet(path, index=False)
    print(f"[cache] saved {name}: {len(df):,} rows")
    return df

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)
print("\nexisting database tables (0 everywhere is normal on a fresh run):")
for t in ["core", "core_retired", "gnaf_key", "retired_key",
          "gnaf_streets", "street_matches"]:
    print(f"  {t:16s} {table_rows(t):>12,}")

duckdb 1.5.5 | pandas 3.0.3

existing database tables (0 everywhere is normal on a fresh run):
  core               15,901,249
  core_retired        1,004,575
  gnaf_key           20,837,941
  retired_key         1,103,257
  gnaf_streets          530,796
  street_matches         25,400


In [ ]:
# Cell 6: build the street-type, suffix and state vocabularies from the
# G-NAF authority tables, build the address regex from them, and test it on
# eight strings.

st_type_path, st_suffix_path, flat_type_path, lvl_type_path = AUT_FILES

def read_aut(path):
    # keep_default_na=False keeps the street type 'NULL', a real G-NAF value
    return pd.read_csv(path, sep="|", dtype=str, keep_default_na=False)

st_type   = read_aut(st_type_path)      # CODE = full word, NAME = abbrev
st_suffix = read_aut(st_suffix_path)    # CODE = abbrev,    NAME = full word
flat_type = read_aut(flat_type_path)
lvl_type  = read_aut(lvl_type_path)

TYPE_MAP = {}
for code, name in zip(st_type.CODE, st_type.NAME):
    TYPE_MAP[code] = code        # STREET -> STREET
    TYPE_MAP[name] = code        # ST     -> STREET

# Abbreviations in common use that G-NAF does not list
TYPE_MAP.update({
    "AVE": "AVENUE", "AVEN": "AVENUE",
    "CRES": "CRESCENT", "CRESC": "CRESCENT",
    "GRV": "GROVE", "GVE": "GROVE",
    "STR": "STREET", "STRT": "STREET",
    "TER": "TERRACE", "TERR": "TERRACE",
    "BLVD": "BOULEVARD", "BLV": "BOULEVARD",
    "PKWY": "PARKWAY", "XING": "CROSSING",
    "RDG": "RIDGE", "SQR": "SQUARE", "HGWY": "HIGHWAY",
    "CIRC": "CIRCUIT", "PARA": "PARADE",
})

SUFFIX_MAP = {}
for code, name in zip(st_suffix.CODE, st_suffix.NAME):
    SUFFIX_MAP[code] = name      # N     -> NORTH
    SUFFIX_MAP[name] = name      # NORTH -> NORTH
SUFFIX_MAP.update({"NTH": "NORTH", "STH": "SOUTH", "EST": "EAST", "WST": "WEST"})

STATE_MAP = {
    "NSW": "NSW", "NEW SOUTH WALES": "NSW",
    "VIC": "VIC", "VICTORIA": "VIC",
    "QLD": "QLD", "QUEENSLAND": "QLD",
    "SA":  "SA",  "SOUTH AUSTRALIA": "SA",
    "WA":  "WA",  "WESTERN AUSTRALIA": "WA",
    "TAS": "TAS", "TASMANIA": "TAS",
    "NT":  "NT",  "NORTHERN TERRITORY": "NT",
    "ACT": "ACT", "AUSTRALIAN CAPITAL TERRITORY": "ACT",
}

def alt(strings):
    """Regex alternation, longest first so STREET beats ST."""
    uniq = sorted({s for s in strings if s}, key=len, reverse=True)
    return "|".join(re.escape(s) for s in uniq)

TYPE_ALT   = alt(TYPE_MAP)
SUFFIX_ALT = alt(SUFFIX_MAP)
LEVEL_ALT  = alt(set(lvl_type.CODE) | set(lvl_type.NAME))
# LOT has its own group, so it is not a flat type
FLAT_ALT   = alt((set(flat_type.CODE) | set(flat_type.NAME)) - {"LOT"})

STATE_ABBREV = ["NSW", "VIC", "QLD", "SA", "WA", "TAS", "NT", "ACT"]
STATE_FULL   = ["NEW SOUTH WALES", "VICTORIA", "QUEENSLAND", "SOUTH AUSTRALIA",
                "WESTERN AUSTRALIA", "TASMANIA", "NORTHERN TERRITORY",
                "AUSTRALIAN CAPITAL TERRITORY"]
STATE_ALT = (r"(?:" + "|".join(sorted(STATE_ABBREV, key=len, reverse=True))
             + r"|(?i:" + "|".join(sorted(STATE_FULL, key=len, reverse=True)) + r"))")

WS = r"[^\S\n]+(?:\n[^\S\n]*)?"    # spaces/tabs, at most one line break

# Each optional prefix starts at \b, so the flat type ANT cannot match the
# end of "Defendant".
LEVEL_PART = r"(?:\b(?P<level>(?i:" + LEVEL_ALT + r")\s*\d{1,3}[A-Za-z]?)[\s,]+)?"
FLAT_PART  = (r"(?:\b(?P<flat_type>(?i:" + FLAT_ALT + r"))\s*)?"
              r"(?:\b(?P<flat>[\dA-Za-z]{1,5})\s*/\s*)?")

# Either "Lot 4" or "Lot 4, 22", or a plain street number.
NUM_BLOCK = (
    r"\b(?:"
      r"(?i:LOTS?)\s*(?P<lot>\d{1,6}[A-Za-z]?)"
      r"(?:[\s,]+(?P<lot_number>\d{1,5}[A-Za-z]?))?"
      r"|"
      r"(?P<number>\d{1,5}[A-Za-z]?(?:\s*[-\u2013]\s*\d{1,5}[A-Za-z]?)?)"
    r")" + WS
)

STREET_PART = (r"(?P<street>(?:[A-Za-z][A-Za-z'\-]*" + WS + r"){1,4}?)"
               r"(?P<st_type>(?i:" + TYPE_ALT + r"))\b"
               r"(?:" + WS + r"(?P<st_suffix>(?i:" + SUFFIX_ALT + r"))\b)?")

TAIL_PART = (r"[\s,]+"
             r"(?P<locality>(?:[A-Za-z][A-Za-z'\-]*[^\S\n]*){1,4}?)"
             r"[\s,]*\b(?P<state>" + STATE_ALT + r")\b"
             r"[\s,]*(?P<postcode>\d{4})\b")

ADDRESS_RE = re.compile(          # case sensitive: "Act 1976" is not ACT
    LEVEL_PART + FLAT_PART + NUM_BLOCK + STREET_PART + TAIL_PART
)

# Any state and postcode: a quick test before running the full regex
PRESCAN = re.compile(r"\b" + STATE_ALT + r"\b[\s,]*\d{4}\b")

print(f"{len(TYPE_MAP)} street-type variants, {len(SUFFIX_MAP)} suffix variants")
for t in [
    "45 Bourke Rd, Alexandria NSW 2015",
    "Lot 4 Green Valley Road, Green Valley NSW 2168",
    "Lot 12, 88 Phillip Street Sydney NSW 2000",
    "Level 23, 525 Collins Street, Melbourne, Victoria 3000",
    "Defendant\n32 Salisbury Street\nBAYSWATER WA 6053",   # must not capture 'ant'
    "Marvel 3, 88 Phillip Street Sydney NSW 2000",          # must not capture level
    "37M of the Federal Court of Australia Act 1976",       # must not match
    "1a Old South Head Road Vaucluse NSW 2030",             # known mis-split
]:
    m = ADDRESS_RE.search(t)
    got = [(k, v) for k, v in m.groupdict().items() if v] if m else "NO MATCH"
    print(f"{t!r}\n    {got}")

488 street-type variants, 41 suffix variants
'45 Bourke Rd, Alexandria NSW 2015'
    [('number', '45'), ('street', 'Bourke '), ('st_type', 'Rd'), ('locality', 'Alexandria '), ('state', 'NSW'), ('postcode', '2015')]
'Lot 4 Green Valley Road, Green Valley NSW 2168'
    [('lot', '4'), ('street', 'Green Valley '), ('st_type', 'Road'), ('locality', 'Green Valley '), ('state', 'NSW'), ('postcode', '2168')]
'Lot 12, 88 Phillip Street Sydney NSW 2000'
    [('lot', '12'), ('lot_number', '88'), ('street', 'Phillip '), ('st_type', 'Street'), ('locality', 'Sydney '), ('state', 'NSW'), ('postcode', '2000')]
'Level 23, 525 Collins Street, Melbourne, Victoria 3000'
    [('level', 'Level 23'), ('number', '525'), ('street', 'Collins '), ('st_type', 'Street'), ('locality', 'Melbourne'), ('state', 'Victoria'), ('postcode', '3000')]
'Defendant\n32 Salisbury Street\nBAYSWATER WA 6053'
    [('number', '32'), ('street', 'Salisbury '), ('st_type', 'Street'), ('locality', 'BAYSWATER '), ('state', 'WA'), ('post

In [ ]:
# Cell 7: the acceptance filter and extract(), tried on the first 1,000
# corpus documents. The filter rejects legislation citations and regex bleed
# (ordinary sentence words captured as part of an address).

from tqdm.auto import tqdm

KEEP_TYPES = {"decision"}
CITE_GUARD = re.compile(
    r"(?i)\b(?:s|ss|section|sections|reg|regs|cl|cll|para|paras|item|sch|div|pt|r|rr)\.?\s*$"
)
JURIS_TAIL = re.compile(r"^\s*\((?:Cth|NSW|Vic|Qld|SA|WA|Tas|NT|ACT)\)")
FILLER     = re.compile(r"(?i)\b(?:of|the|and|under|pursuant|for|to|in)\b")

def accept(m, text):
    g = m.groupdict()
    if CITE_GUARD.search(text[max(0, m.start() - 30):m.start()]):
        return False
    if JURIS_TAIL.match(text[m.end():m.end() + 12]):
        return False
    if FILLER.search(g["street"] or ""):
        return False
    return True

def extract(doc):
    """All accepted addresses in one document, as dicts."""
    txt = doc.get(TEXT_FIELD) or ""
    if not PRESCAN.search(txt):
        return []
    rows = []
    for m in ADDRESS_RE.finditer(txt):
        if not accept(m, txt):
            continue
        g = m.groupdict()
        g["raw"]          = m.group(0)
        g["citation"]     = doc.get("citation")
        g["jurisdiction"] = doc.get("jurisdiction")
        g["doc_type"]     = doc.get("type")
        g["url"]          = doc.get("url")
        rows.append(g)
    return rows

SMOKE_N = 1_000
hits, docs, skipped = [], 0, 0
t0 = time.time()

with CORPUS.open("rb") as f:
    pbar = tqdm(total=SMOKE_N, unit="doc", desc="smoke test")
    for line in f:
        doc = loads(line)
        docs += 1; pbar.update(1)
        if doc.get("type") not in KEEP_TYPES:
            skipped += 1
        else:
            hits.extend(extract(doc))
        pbar.set_postfix(addr=len(hits))
        if docs >= SMOKE_N:
            break
    pbar.close()

print(f"{docs:,} docs ({skipped:,} skipped), {len(hits):,} addresses, {time.time()-t0:.1f}s")
print(f"projected full run: ~{(time.time()-t0) * 232_560 / docs / 60:.0f} min")
pd.DataFrame(hits).sample(min(25, len(hits)))[["raw", "citation"]]

1,000 docs (639 skipped), 26 addresses, 1.6s
projected full run: ~6 min


,raw,citation
10,"158-160 Adderley Street West, Auburn NSW 2144","Luxtown Pty Ltd (Administrators Appointed), in..."
5,"Level 8, 25 Grenfell Street, Adelaide, South A...",SPLB v Minister for Immigration & Multicultura...
12,"7 Koonya Circuit, Caringbah NSW 2229","Luxtown Pty Ltd (Administrators Appointed), in..."
8,"154 Elizabeth Street, Sydney NSW 2000",Law Society of NSW v Berger (No. 1) [2017] NSW...
13,"6 Victoria Avenue, Castle Hill NSW 2151","Luxtown Pty Ltd (Administrators Appointed), in..."
24,"Level 6, 18 Mount Street, Perth WA 6000",Genovese v BGC Construction Pty Ltd [2005] FCA...
17,"11 Patrick Street, Campbelltown NSW 2560",Mathews v Health Insurance Commission (No. 3)[...
9,"321 Parramatta Road, Auburn NSW 2144","Luxtown Pty Ltd (Administrators Appointed), in..."
2,530 Collins Street\nMELBOURNE VIC 3000,Australian Competition & Consumer Commission v...
22,32 Salisbury Street\nBAYSWTER WA 6053,Genovese v BGC Construction Pty Ltd [2005] FCA...


In [ ]:
# Cell 8: full harvest of the corpus. Streams the corpus, extracts addresses
# from decision documents and writes them to harvest/ in chunks. The byte
# offset is saved to _progress.json only after each chunk is on disk, so an
# interrupted run resumes where it stopped. Delete harvest/ to start again.

import pyarrow as pa, pyarrow.parquet as pq

COLUMNS = ["lot", "lot_number", "level", "flat_type", "flat", "number",
           "street", "st_type", "st_suffix", "locality", "state", "postcode",
           "raw", "citation", "jurisdiction", "doc_type", "url"]
SCHEMA = pa.schema([(c, pa.string()) for c in COLUMNS])

FLUSH_EVERY_DOCS = 5_000
TOTAL_BYTES      = CORPUS.stat().st_size
TYPE_RE          = re.compile(rb'"type"\s*:\s*"([^"]+)"')

def save_state(d):
    tmp = HARVEST_PROGRESS.with_suffix(".tmp")
    tmp.write_text(stdjson.dumps(d))
    os.replace(tmp, HARVEST_PROGRESS)    # atomic, never a half-written file

st = (stdjson.loads(HARVEST_PROGRESS.read_text()) if HARVEST_PROGRESS.exists()
      else {"bytes": 0, "docs": 0, "chunk": 0, "addrs": 0})
if st["bytes"]:
    print(f"resuming at doc {st['docs']:,} ({st['addrs']:,} addresses so far)")

buf = []
def flush():
    global buf
    if buf:
        pq.write_table(pa.Table.from_pylist(buf, schema=SCHEMA),
                       HARVEST / f"part-{st['chunk']:05d}.parquet")
        st["chunk"] += 1
        st["addrs"] += len(buf)
        buf = []
    save_state(st)                       # state saved only after the data

f = CORPUS.open("rb")
f.seek(st["bytes"])
pbar = tqdm(total=TOTAL_BYTES, initial=st["bytes"],
            unit="B", unit_scale=True, desc="harvest")
try:
    while True:
        line = f.readline()
        if not line:
            break
        st["bytes"] += len(line); st["docs"] += 1
        pbar.update(len(line))

        # byte-level type check avoids JSON-parsing most of the corpus
        tm = TYPE_RE.search(line, 0, 800)
        if tm is None or tm.group(1).decode(errors="ignore") in KEEP_TYPES:
            doc = loads(line)
            if doc.get("type") in KEEP_TYPES:
                for g in extract(doc):
                    buf.append({c: g.get(c) for c in COLUMNS})

        if st["docs"] % FLUSH_EVERY_DOCS == 0:
            flush()
            pbar.set_postfix(addr=f"{st['addrs']:,}")
finally:
    flush()                              # also runs on kernel interrupt
    f.close(); pbar.close()

print(f"done: {st['addrs']:,} addresses from {st['docs']:,} documents")

resuming at doc 232,560 (8,707 addresses so far)


done: 8,707 addresses from 232,560 documents


In [ ]:
# Cell 9: load G-NAF Core into the database and build the match keys, one
# per house number and one per lot number, plus gnaf_streets (the distinct
# streets scanned in cell 12). Skipped when the tables exist, unless "gnaf"
# is in FORCE. ALIAS_PRINCIPAL holds 'PRINCIPAL' or 'ALIAS' in full, so a
# join on 'P' would return no rows and raise no error.

NEEDED = ["core", "core_retired", "gnaf_key", "retired_key", "gnaf_streets"]

if all(table_rows(t) > 0 for t in NEEDED) and "gnaf" not in FORCE:
    print("[cache] G-NAF tables already built, skipping ingest")
    for t in NEEDED:
        print(f"  {t:16s} {table_rows(t):>12,}")
else:
    for p in sorted(CORE.glob("*.psv")):
        print(f"{p.stat().st_size/1e9:6.2f} GB  {p.name}")

    # nullstr='' keeps the street type 'NULL' as a value
    con.execute(f"""
    CREATE OR REPLACE TABLE core_all AS
    SELECT * FROM read_csv('{CORE}/*.psv',
        delim='|', header=true, all_varchar=true, nullstr='', filename=true);
    """)
    print(con.sql("""
      SELECT regexp_extract(filename, '[^/]+$') AS file, COUNT(*) n
      FROM core_all GROUP BY 1 ORDER BY n DESC
    """).df())

    con.execute("""
    CREATE OR REPLACE TABLE core AS
    SELECT * FROM core_all WHERE NOT upper(filename) LIKE '%RETIRED%';
    CREATE OR REPLACE TABLE core_retired AS
    SELECT * FROM core_all WHERE upper(filename) LIKE '%RETIRED%';
    """)

    KEYS_SQL = """
    CREATE OR REPLACE TABLE {out} AS
    WITH b AS (
      SELECT ADDRESS_DETAIL_PID AS pid, ADDRESS_LABEL,
             ALIAS_PRINCIPAL, PRINCIPAL_PID,
             POSTCODE                                AS postcode,
             UPPER(STATE)                            AS state_n,
             UPPER(TRIM(LOCALITY_NAME))              AS locality_n,
             UPPER(TRIM(STREET_NAME))                AS street_n,
             UPPER(TRIM(STREET_TYPE))                AS type_n,
             UPPER(TRIM(COALESCE(STREET_SUFFIX,''))) AS suffix_n,
             UPPER(TRIM(COALESCE(NUMBER_FIRST,'')))  AS number_n,
             UPPER(TRIM(COALESCE(LOT_NUMBER,'')))    AS lot_n
      FROM {src}
    )
    SELECT *, regexp_replace(UPPER(CONCAT_WS(' ', k, street_n, type_n,
                  suffix_n, locality_n, state_n, postcode)), '\\s+', ' ', 'g') AS match_key
    FROM (
      SELECT b.*, number_n AS k FROM b WHERE number_n <> ''
      UNION ALL
      SELECT b.*, 'LOT ' || lot_n AS k FROM b WHERE lot_n <> ''
    );
    """
    con.execute(KEYS_SQL.format(out="gnaf_key",    src="core"))
    con.execute(KEYS_SQL.format(out="retired_key", src="core_retired"))

    con.execute("""
    CREATE OR REPLACE TABLE gnaf_streets AS
    SELECT DISTINCT postcode, state_n, locality_n, street_n, type_n, suffix_n
    FROM gnaf_key;
    """)
    con.execute("CREATE INDEX IF NOT EXISTS ix_mk ON gnaf_key(match_key);")
    con.execute("CREATE INDEX IF NOT EXISTS ix_rk ON retired_key(match_key);")
    con.execute("CREATE INDEX IF NOT EXISTS ix_pc ON gnaf_key(postcode);")
    con.execute("DROP TABLE IF EXISTS core_all;")   # about 17M rows, no longer needed
    print("built.")

print(f"\n{table_rows('gnaf_streets'):,} distinct streets, "
      f"{table_rows('gnaf_key'):,} match keys")

[cache] G-NAF tables already built, skipping ingest
  core               15,901,249
  core_retired        1,004,575
  gnaf_key           20,837,941
  retired_key         1,103,257
  gnaf_streets          530,796

530,796 distinct streets, 20,837,941 match keys


In [ ]:
# Cell 9b: two side tables from core. gnaf_range holds NUMBER_LAST for ranged
# addresses, so "205-207" can be rebuilt. loc_alias holds each locality that
# G-NAF records as an alias of another. Both are built only when missing or
# when "gnaf" is in FORCE, because DuckDB does not give back the space of a
# replaced table and the database file would grow on every run.

_have_both = table_rows("gnaf_range") > 0 and table_rows("loc_alias") > 0
_rebuild   = ("gnaf" in FORCE) or not _have_both
print("[build] gnaf_range and loc_alias" if _rebuild
      else "[cache] gnaf_range and loc_alias already exist, not rebuilt")

# 1. Number ranges
if _rebuild: con.execute("""
CREATE OR REPLACE TABLE gnaf_range AS
SELECT ADDRESS_DETAIL_PID                        AS pid,
       UPPER(TRIM(COALESCE(NUMBER_FIRST, '')))   AS number_first,
       UPPER(TRIM(COALESCE(NUMBER_LAST , '')))   AS number_last
FROM core
WHERE COALESCE(TRIM(NUMBER_LAST), '') <> ''
  AND COALESCE(TRIM(NUMBER_FIRST), '') <> '';
""")
n_rng = table_rows("gnaf_range")
n_key = table_rows("gnaf_key")
print(f"gnaf_range: {n_rng:,} ranged addresses "
      f"= {n_rng / max(n_key, 1) * 100:.2f}% of the {n_key:,} match keys\n")
print(con.sql("""
    SELECT number_first || '-' || number_last AS range, count(*) n
    FROM gnaf_range GROUP BY 1 ORDER BY n DESC LIMIT 8
""").df().to_string(index=False), "\n")

# 2. Locality aliases. An alias address points at a principal address. Where
# the two locality names differ, the alias name is an unofficial name for the
# principal locality. LIKE 'A%' matches the alias flag stored as 'ALIAS' or 'A'.
probe = con.sql("""
    SELECT ALIAS_PRINCIPAL, count(*) n FROM core GROUP BY 1 ORDER BY n DESC
""").df()
print("ALIAS_PRINCIPAL values in core:")
print(probe.to_string(index=False), "\n")

if _rebuild: con.execute("""
CREATE OR REPLACE TABLE loc_alias AS
SELECT DISTINCT
       UPPER(TRIM(a.LOCALITY_NAME)) AS alias_loc,
       UPPER(TRIM(p.LOCALITY_NAME)) AS principal_loc,
       UPPER(TRIM(p.STATE))         AS state,
       p.POSTCODE                   AS postcode
FROM core a
JOIN core p ON a.PRINCIPAL_PID = p.ADDRESS_DETAIL_PID
WHERE UPPER(COALESCE(a.ALIAS_PRINCIPAL, '')) LIKE 'A%'
  AND UPPER(TRIM(a.LOCALITY_NAME)) <> UPPER(TRIM(p.LOCALITY_NAME));
""")
n_alias = table_rows("loc_alias")
print(f"loc_alias: {n_alias:,} distinct (unofficial -> gazetted) locality pairs")
if n_alias:
    print(con.sql("""
        SELECT alias_loc, principal_loc, state, count(*) n
        FROM loc_alias GROUP BY 1,2,3 ORDER BY n DESC LIMIT 15
    """).df().to_string(index=False))
else:
    print("  none found. Set alias_table='' in the synthesis notebook's COLMAP;\n"
          "  err_loc_sublocal will fall back to its pattern model.")

In [ ]:
# Cell 10: normalise the harvested addresses to G-NAF's conventions,
# exact-match them against live and retired G-NAF, and give each unmatched
# address a first-pass error class. Only unmatched addresses go on to the
# rest of the notebook. Every field passes through _s() first.

def canon_type(s):   return TYPE_MAP.get(_s(s).upper().strip(), _s(s).upper().strip())
def canon_suffix(s): return SUFFIX_MAP.get(_s(s).upper().strip(), "")
def canon_state(s):  return STATE_MAP.get(_s(s).upper().strip(), _s(s).upper().strip())
def squash(s):       return re.sub(r"\s+", " ", _s(s).upper()).strip()

def build_diag():
    parts = sorted(HARVEST.glob(HARVEST_PARTS))
    if not parts:
        raise FileNotFoundError(
            f"No {HARVEST_PARTS} in {HARVEST.resolve()}. Run cell 8 first.")
    cand = pd.read_parquet(parts)

    for col in ["lot", "lot_number", "number", "street", "st_type", "st_suffix",
                "locality", "state", "postcode", "citation", "url", "raw"]:
        if col not in cand.columns:
            cand[col] = ""
        cand[col] = cand[col].astype("string").fillna("")
    print(f"  {len(cand):,} raw matches")

    cand["street_n"]   = cand["street"].map(squash)
    cand["type_n"]     = cand["st_type"].map(canon_type)
    cand["suffix_n"]   = cand["st_suffix"].map(canon_suffix)
    cand["locality_n"] = cand["locality"].map(squash)
    cand["state_n"]    = cand["state"].map(canon_state)
    cand["lot_n"]      = cand["lot"].map(squash)
    cand["number_n"]   = cand.apply(
        lambda r: squash(_s(r["number"]) or _s(r["lot_number"])).split("-")[0], axis=1)

    bad = (cand.number_n == "") & (cand.lot_n == "")
    cand = cand[~bad].reset_index(drop=True)

    # A range like "12-14" matches G-NAF only if one end exists on the street.
    # Test both ends and keep whichever is real (low end wins a tie).
    rng = cand["number"].str.extract(
        r"^\s*(\d{1,5}[A-Za-z]?)\s*[-\u2013]\s*(\d{1,5}[A-Za-z]?)\s*$")
    cand["num_lo"] = rng[0].fillna("").str.upper()
    cand["num_hi"] = rng[1].fillna("").str.upper()
    cand["row_id"] = range(len(cand))
    con.register("cand_r", cand)

    resolved = con.sql("""
    SELECT c.row_id, c.num_lo, c.num_hi,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = c.postcode
                AND g.street_n = c.street_n AND g.k = c.num_lo) AS lo_ok,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = c.postcode
                AND g.street_n = c.street_n AND g.k = c.num_hi) AS hi_ok
    FROM cand_r c WHERE c.num_lo <> '';
    """).df()

    pick = {}
    for r in resolved.itertuples():
        if r.lo_ok and not r.hi_ok:   pick[r.row_id] = r.num_lo
        elif r.hi_ok and not r.lo_ok: pick[r.row_id] = r.num_hi
        elif r.lo_ok and r.hi_ok:     pick[r.row_id] = r.num_lo
    cand["number_resolved"] = cand["row_id"].map(pick).fillna(cand["number_n"])

    cand["k"] = cand.apply(
        lambda r: r["number_resolved"] if r["number_resolved"] else f"LOT {r['lot_n']}",
        axis=1)
    cand["match_key"] = (cand.k + " " + cand.street_n + " " + cand.type_n + " "
                         + cand.suffix_n + " " + cand.locality_n + " "
                         + cand.state_n + " " + cand.postcode).map(squash)
    cand = cand.drop_duplicates("match_key").reset_index(drop=True)
    print(f"  {len(cand):,} distinct addresses")
    con.register("cand", cand)

    con.execute("""
    CREATE OR REPLACE TABLE unmatched AS
    SELECT c.*, r.ADDRESS_LABEL AS retired_label
    FROM cand c
    LEFT JOIN (SELECT DISTINCT match_key, pid FROM gnaf_key) g USING (match_key)
    LEFT JOIN (SELECT match_key, first(ADDRESS_LABEL) AS ADDRESS_LABEL
               FROM retired_key GROUP BY match_key) r USING (match_key)
    WHERE g.pid IS NULL;
    """)

    out = con.sql("""
    SELECT u.raw, u.match_key, u.citation, u.url, u.retired_label,
           u.postcode, u.state_n, u.locality_n, u.street_n,
           u.type_n, u.suffix_n, u.k, u.number_resolved AS number_n, u.lot_n,
      (u.state_n || ' ' || u.postcode) IN
          (SELECT DISTINCT state_n || ' ' || postcode FROM gnaf_key)      AS sp_ok,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = u.postcode)     AS pc_exists,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = u.postcode
                AND g.locality_n = u.locality_n)                          AS loc_ok,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = u.postcode
                AND g.street_n = u.street_n)                              AS street_ok,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = u.postcode
                AND g.street_n = u.street_n AND g.type_n = u.type_n)      AS type_ok,
      EXISTS (SELECT 1 FROM gnaf_key g WHERE g.postcode = u.postcode
                AND g.street_n = u.street_n AND g.k = u.k)                AS number_ok
    FROM unmatched u;
    """).df()

    out["retired_label"] = out["retired_label"].map(_s)
    for c in ["sp_ok", "pc_exists", "loc_ok", "street_ok", "type_ok", "number_ok"]:
        out[c] = out[c].fillna(False).astype(bool)

    def label(r):
        if r.retired_label:                  return "RETIRED"
        if not r.pc_exists:                  return "PPC"
        if not r.sp_ok:                      return "PPC-STATE"
        if r.loc_ok and not r.street_ok:     return "STN"
        if r.street_ok and not r.loc_ok:     return "LOC"
        if not r.loc_ok and not r.street_ok: return "LOC-STN"
        if r.street_ok and not r.type_ok:    return "STT"
        if not r.number_ok:                  return "NOF"
        return "UNKNOWN"

    out["change_type"] = out.apply(label, axis=1)
    return out

diag = cached("diag", build_diag, force="diag" in FORCE)
print(diag.change_type.value_counts())

[cache] loaded diag: 1,045 rows
change_type
LOC          222
STN          202
NOF          184
LOC-STN      147
UNKNOWN      133
RETIRED       63
PPC           52
STT           40
PPC-STATE      2
Name: count, dtype: int64


In [ ]:
# Cell 10b: every plausible reading of a written house number, used in
# cell 14. Covers slash forms (75/77), ranges (123-130) and a trailing
# letter (12A).

def get_number_readings(parsed_k, raw_str):
    """Candidate house-number strings from the parsed and raw inputs."""
    candidates = set()
    for source in [parsed_k, str(raw_str).split()[0] if raw_str else ""]:
        w = str(source).upper().replace("\u2013", "-").replace("\u2014", "-").strip()
        if not w:
            continue
        candidates.add(w)
        if "/" in w:
            candidates.add(w.split("/")[-1])  # Unit 75 at number 77 -> 77
            candidates.add(w.split("/")[0])   # 75/77 -> 75
        if "-" in w:
            candidates.add(w.split("-")[0])   # range start: 123-130 -> 123
        
        # digits only: 12A -> 12
        digits = re.sub(r"[^0-9]", "", w)
        if digits:
            candidates.add(digits)
            
    return [c for c in candidates if c]

In [ ]:
# Cell 11: drop rows the regex should not have captured. Redactions are runs
# of x standing in for suppressed names. Bleed is ordinary sentence words
# (of, the, lived, court) in the street or suburb field. Connectives such as
# "via" are kept, but the locality is cut at them: "SMITHFIELD VIA CAIRNS" is
# a real address in SMITHFIELD.

REDACT = re.compile(r"(?i)(?:^|\s)x{1,12}(?:$|\s)")
BLEED  = re.compile(
    r"(?i)\b(?:of|the|and|under|pursuant|for|to|in|at|by|with|from|or|"
    r"is|was|were|be|been|has|had|have|that|which|said|she|he|they|"
    r"lived|resided|situated|known|being|court|supreme)\b")
CONNECTIVE = re.compile(r"(?i)\b(?:via|in|at|near|c/-|c/|care of)\b")

def build_to_match():
    d = diag.copy()
    d["is_redacted"] = (d.street_n.str.contains(REDACT, na=False)
                        | d.locality_n.str.contains(REDACT, na=False))
    d["is_bleed"]    = (d.street_n.str.contains(BLEED, na=False)
                        | d.locality_n.str.contains(BLEED, na=False))
    d["has_connective"] = d.raw.str.contains(CONNECTIVE, na=False)
    print(f"  redacted {d.is_redacted.sum():,} | bleed {d.is_bleed.sum():,} "
          f"| connective {d.has_connective.sum():,} (kept)")
    d["locality_clean"] = (d.locality_n
        .str.replace(r"(?i)\s+(?:VIA|IN|AT|NEAR)\s+.*$", "", regex=True)
        .str.replace(r"(?i)\s+(?:VIA|IN|AT|NEAR)$", "", regex=True)
        .str.strip())
    return d[~(d.is_redacted | d.is_bleed)].reset_index(drop=True)

to_match = cached("to_match", build_to_match, force="clean" in FORCE)

[cache] loaded to_match: 1,016 rows


In [ ]:
# Cell 12: score every cleaned address against every distinct G-NAF street
# and keep the top 25. The score is a weighted sum of street and locality
# similarity, street type and postcode, so a strong locality can outweigh a
# slightly weaker street. A locality written as whole words of the G-NAF one
# (TUCKI for TUCKI TUCKI) scores 1.0, so it beats a similar spelling
# (TUCKURIMBA). Resumable. Put "exhaustive" in FORCE to start over.

import time
from tqdm.auto import tqdm

TOP_K, BATCH = 25, 200
W_STREET, W_LOC, W_TYPE, W_PC = 0.45, 0.35, 0.10, 0.10

if "exhaustive" in FORCE:
    con.execute("DROP TABLE IF EXISTS street_matches;")

con.execute("""
CREATE TABLE IF NOT EXISTS street_matches (
    raw VARCHAR, rank INTEGER, sim DOUBLE,
    street_sim DOUBLE, loc_sim DOUBLE,
    g_postcode VARCHAR, g_state VARCHAR, g_locality VARCHAR,
    g_street VARCHAR, g_type VARCHAR, g_suffix VARCHAR);""")

# Clear any address left partially written by an interrupt
con.execute(f"""DELETE FROM street_matches WHERE raw IN (
    SELECT raw FROM street_matches GROUP BY raw HAVING count(*) < {TOP_K});""")

done = set(con.execute("SELECT DISTINCT raw FROM street_matches").df()["raw"]) \
       if table_rows("street_matches") else set()
todo = to_match[~to_match.raw.isin(done)].reset_index(drop=True)
print(f"{len(done):,} already matched, {len(todo):,} to go")

SQL = f"""
INSERT INTO street_matches
SELECT raw, rn, sim, street_sim, loc_sim,
       postcode, state_n, locality_n, street_n, type_n, suffix_n
FROM (
  SELECT b.raw, g.postcode, g.state_n, g.locality_n,
         g.street_n, g.type_n, g.suffix_n,
         jaro_winkler_similarity(b.street_n, g.street_n) AS street_sim,
         
         -- Whole-word aware locality similarity
         CASE
           WHEN contains(' ' || g.locality_n || ' ', ' ' || b.locality_clean || ' ')
             OR contains(' ' || b.locality_clean || ' ', ' ' || g.locality_n || ' ')
           THEN 1.0
           ELSE jaro_winkler_similarity(b.locality_clean, g.locality_n)
         END AS loc_sim,
         
         {W_STREET} * jaro_winkler_similarity(b.street_n, g.street_n)
       + {W_LOC}    * CASE
                        WHEN contains(' ' || g.locality_n || ' ', ' ' || b.locality_clean || ' ')
                          OR contains(' ' || b.locality_clean || ' ', ' ' || g.locality_n || ' ')
                        THEN 1.0
                        ELSE jaro_winkler_similarity(b.locality_clean, g.locality_n)
                      END
       + {W_TYPE}   * CAST(b.type_n = g.type_n AS INT)
       + {W_PC}     * CAST(b.postcode = g.postcode AS INT) AS sim,
         
         ROW_NUMBER() OVER (
            PARTITION BY b.raw
            ORDER BY {W_STREET} * jaro_winkler_similarity(b.street_n, g.street_n)
                   + {W_LOC}    * CASE
                                    WHEN contains(' ' || g.locality_n || ' ', ' ' || b.locality_clean || ' ')
                                      OR contains(' ' || b.locality_clean || ' ', ' ' || g.locality_n || ' ')
                                    THEN 1.0
                                    ELSE jaro_winkler_similarity(b.locality_clean, g.locality_n)
                                  END
                   + {W_TYPE}   * CAST(b.type_n = g.type_n AS INT)
                   + {W_PC}     * CAST(b.postcode = g.postcode AS INT) DESC,
                     CAST(b.type_n = g.type_n AS INT) DESC) AS rn
  FROM batch b CROSS JOIN gnaf_streets g
) WHERE rn <= {TOP_K};
"""

if len(todo):
    t0 = time.time()
    try:
        for i in tqdm(range(0, len(todo), BATCH), desc="exhaustive match"):
            con.register("batch", todo.iloc[i:i + BATCH][
                ["raw", "street_n", "locality_clean", "type_n", "postcode"]])
            con.execute(SQL)
    finally:
        n = table_rows("street_matches")
        print(f"\n{n:,} pairs, {n // TOP_K:,} addresses done, "
              f"{time.time() - t0:.0f}s this run")
else:
    print("[cache] nothing to do")

0 already matched, 1,016 to go



25,400 pairs, 1,016 addresses done, 173s this run


In [ ]:
# Cell 13: print a fixed random sample of 30 addresses with their top five
# candidates, so a person can check by hand whether the correct address is
# among them and at what rank.

SAMPLE_N, SHOW_K = 30, 5

ids = (con.sql("SELECT DISTINCT raw FROM street_matches").df()
         .sample(SAMPLE_N, random_state=0)["raw"].tolist())
con.register("sample_ids", pd.DataFrame({"raw": ids}))

rows = con.sql(f"""
    SELECT s.raw, m.rank, ROUND(m.sim, 3) AS sim,
           m.g_street || ' ' || m.g_type
             || CASE WHEN m.g_suffix <> '' THEN ' ' || m.g_suffix ELSE '' END
             || ' ' || m.g_locality || ' ' || m.g_state || ' ' || m.g_postcode AS cand
    FROM sample_ids s JOIN street_matches m USING (raw)
    WHERE m.rank <= {SHOW_K}
    ORDER BY s.raw, m.rank
""").df()

for i, (raw, grp) in enumerate(rows.groupby("raw", sort=False), 1):
    print(f"\n[{i:>2}] {re.sub(r'\\s+', ' ', raw).strip()}")
    for r in grp.itertuples():
        print(f"       {r.rank}. {r.sim}  {r.cand}")

print("\n" + "="*70)
print("For each: is the correct address in the list? At what rank?")
print("Tally hits at rank 1, within 5, and misses.")


[ 1] 123-130 South Terrace, Adelaide, SA, 5000
       1. 1.0  SOUTH TERRACE ADELAIDE SA 5000
       2. 0.88  NORTH TERRACE ADELAIDE SA 5000
       3. 0.86  SOUTHCOTT WALK ADELAIDE SA 5000
       4. 0.8  SOUTH TERRACE LAUDERDALE TAS 7021
       5. 0.795  WAYMOUTH PLACE ADELAIDE SA 5000

[ 2] 13 Cullya Road, Orange, NSW 2800
       1. 0.874  CULLYA ROAD CLERGATE NSW 2800
       2. 0.865  ICELY ROAD ORANGE NSW 2800
       3. 0.85  BULGAS ROAD ORANGE NSW 2800
       4. 0.839  HUNTLEY ROAD ORANGE NSW 2800
       5. 0.839  ALLENBY ROAD ORANGE NSW 2800

[ 3] 13/12 Minjungbai Drive, Tweed Heads South NSW 2486
       1. 0.982  MINJUNGBAL DRIVE TWEED HEADS SOUTH NSW 2486
       2. 0.801  INDUSTRY DRIVE TWEED HEADS SOUTH NSW 2486
       3. 0.797  TRIGONIE DRIVE TWEED HEADS SOUTH NSW 2486
       4. 0.795  MACHINERY DRIVE TWEED HEADS SOUTH NSW 2486
       5. 0.789  CHAMPAGNE DRIVE TWEED HEADS SOUTH NSW 2486

[ 4] 166 Georges River Road, NSW 2133
       1. 0.742  GEORGES RIVER ROAD CROYDON PARK NSW

In [ ]:
# Cell 14: for each candidate street, check whether the written house number
# exists on it in G-NAF, either exactly (pass A, principal records only) or
# inside a numbered range on the same odd or even side (pass B, principal or
# alias records). Ties go to an exact match, then a principal record, then
# the narrower range, then the lower pid.

def build_resolved():
    # Every reading of each written number (cell 10b)
    tm_df = to_match[["raw", "k", "street_n", "locality_clean",
                      "type_n", "postcode", "state_n",
                      "change_type", "retired_label"]].copy()

    reading_rows = []
    for row in tm_df.itertuples():
        reads = get_number_readings(row.k, row.raw)
        for r in reads:
            reading_rows.append({"raw": row.raw, "reading": r})

    readings_df = pd.DataFrame(reading_rows).drop_duplicates()

    con.register("tm", tm_df)
    con.register("readings_df", readings_df)

    return con.sql("""
    WITH cand AS (
      SELECT m.raw, m.rank, m.sim, m.street_sim, m.loc_sim,
             m.g_postcode, m.g_state, m.g_locality,
             m.g_street, m.g_type, m.g_suffix,
             t.k AS orig_k, t.street_n, t.locality_clean, t.type_n,
             t.postcode AS src_postcode, t.state_n AS src_state,
             t.change_type, t.retired_label
      FROM street_matches m
      JOIN tm t USING (raw)
    ),
    matched_keys AS (
      -- Pass A: direct match on any candidate reading. Unchanged.
      SELECT c.raw, c.rank, g.ADDRESS_LABEL, g.pid, g.ALIAS_PRINCIPAL,
             g.k AS matched_k,
             1 AS match_rank,                     -- exact always beats range
             0 AS principal_rank,                 -- Pass A is already PRINCIPAL-only
             0 AS range_width
      FROM cand c
      JOIN readings_df rd ON rd.raw = c.raw
      JOIN gnaf_key g
        ON  g.postcode   = c.g_postcode
        AND g.street_n   = c.g_street
        AND g.type_n     = c.g_type
        AND g.locality_n = c.g_locality
        AND g.ALIAS_PRINCIPAL = 'PRINCIPAL'
        AND g.k          = rd.reading

      UNION ALL

      -- Pass B: written number falls INSIDE a G-NAF range (e.g. 125 inside
      -- 119-125). No ALIAS_PRINCIPAL filter here (see the cell header).
      SELECT c.raw, c.rank, g.ADDRESS_LABEL, g.pid, g.ALIAS_PRINCIPAL,
             (r.number_first || '-' || r.number_last) AS matched_k,
             2 AS match_rank,
             CASE WHEN g.ALIAS_PRINCIPAL = 'PRINCIPAL' THEN 0 ELSE 1 END AS principal_rank,
             (TRY_CAST(r.number_last AS INT) - TRY_CAST(r.number_first AS INT)) AS range_width
      FROM cand c
      JOIN readings_df rd ON rd.raw = c.raw
      JOIN gnaf_key g
        ON  g.postcode   = c.g_postcode
        AND g.street_n   = c.g_street
        AND g.type_n     = c.g_type
        AND g.locality_n = c.g_locality
      JOIN gnaf_range r ON r.pid = g.pid AND r.number_first = g.k
      WHERE TRY_CAST(rd.reading AS INT) IS NOT NULL
        AND TRY_CAST(r.number_first AS INT) IS NOT NULL
        AND TRY_CAST(r.number_last AS INT) IS NOT NULL
        AND TRY_CAST(rd.reading AS INT) BETWEEN TRY_CAST(r.number_first AS INT)
                                            AND TRY_CAST(r.number_last AS INT)
        AND (
              -- range spans both parities (unexpected) -> skip the check
              (TRY_CAST(r.number_first AS INT) % 2) <> (TRY_CAST(r.number_last AS INT) % 2)
              -- normal case: reading must share the range's parity
              OR (TRY_CAST(rd.reading AS INT) % 2) = (TRY_CAST(r.number_first AS INT) % 2)
            )
    )
    SELECT c.*,
           m.ADDRESS_LABEL   AS gnaf_label,
           m.pid             AS gnaf_pid,
           m.ALIAS_PRINCIPAL AS gnaf_alias_principal,
           m.matched_k       AS resolved_k,
           m.ADDRESS_LABEL IS NOT NULL AS number_found
    FROM cand c
    LEFT JOIN matched_keys m ON m.raw = c.raw AND m.rank = c.rank
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY c.raw, c.rank
        ORDER BY m.match_rank, m.principal_rank, m.range_width, m.pid
    ) = 1;
    """).df()

resolved = cached("resolved", build_resolved, force="resolve" in FORCE)
resolved["number_found"] = resolved["number_found"].fillna(False).astype(bool)

resolved.to_parquet(ALL_CANDIDATES_PQ, index=False)
print(f"wrote {len(resolved):,} candidate rows to {ALL_CANDIDATES_PQ}")
print(f"candidates with house number confirmed: {int(resolved.number_found.sum()):,}")

[build] resolved ...
[cache] saved resolved: 25,400 rows
wrote 25,400 candidate rows to output/resolved_all_candidates.parquet
candidates with house number confirmed: 7,093


In [ ]:
# Cell 15: pick one candidate per address by tier. RESOLVED: street, place
# and house number agree. STREET_ONLY: street and place, no number.
# LOCALITY_SHIFT: street only, same state. WEAK_MATCH: number found on a
# weaker street. UNRESOLVABLE: none of these. The street ranks ahead of the
# number, and the street type is a flag, not a tier condition.

STRONG_ST, STRONG_LOC = 0.90, 0.90

r = resolved.copy()
r["number_found"] = r["number_found"].fillna(False).astype(bool)

# Guard against a mis-parsed street_n scoring spuriously high
r["st_sane"] = (r.street_n.fillna("").str.len() >= 3) & \
               (~r.street_n.fillna("").str.fullmatch(r"\d+"))

r["same_street"] = (r.street_sim >= STRONG_ST) & r.st_sane
r["type_differs"] = (r.type_n != r.g_type) & (r.type_n != "") & (r.g_type != "")

r["loc_close"]      = r.loc_sim >= STRONG_LOC
r["pc_state_match"] = (r.src_postcode == r.g_postcode) & (r.src_state == r.g_state)
r["same_place"]     = r.loc_close | r.pc_state_match
# Same postcode but a different suburb name: a colloquial name or a real
# locality error.
r["locality_differs"] = r.pc_state_match & ~r.loc_close
r["cross_state"]      = r.src_state != r.g_state

r["tier"] = np.select(
    [r.same_street & r.same_place & r.number_found,   # 1 street+place+number
     r.same_street & r.same_place,                    # 2 street+place, no number
     r.same_street & ~r.cross_state,                  # 3 street ok, wrong suburb
     r.number_found],                                 # 4 number ok, street weak
    [1, 2, 3, 4], default=5)

best = (r.sort_values(["tier", "rank", "sim"], ascending=[True, True, False])
          .drop_duplicates("raw", keep="first")       # whole rows, unlike groupby().first()
          .reset_index(drop=True))

STATUS = {1: "RESOLVED", 2: "STREET_ONLY", 3: "LOCALITY_SHIFT",
          4: "WEAK_MATCH", 5: "UNRESOLVABLE"}
best["status"] = best.tier.map(STATUS)

print(best.status.value_counts(), "\n")
print(pd.crosstab(best.change_type, best.status), "\n")
print(f"RESOLVED with a differing suburb name : "
      f"{int(best[best.status=='RESOLVED'].locality_differs.sum())}")
print(f"RESOLVED with a differing street type : "
      f"{int(best[best.status=='RESOLVED'].type_differs.sum())}")
print(f"RESOLVED cross-state                  : "
      f"{int(best[best.status=='RESOLVED'].cross_state.sum())}")

best = best.drop(columns=["st_sane", "same_street", "same_place",
                          "loc_close", "pc_state_match"])

for st in ["RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT", "WEAK_MATCH", "UNRESOLVABLE"]:
    sub = best[best.status == st]
    print(f"\n── {st} ({len(sub):,}) " + "─" * 38)
    for x in sub.head(6).itertuples():
        flags = ("T" if x.type_differs else ".") + ("L" if x.locality_differs else ".") + ("X" if x.cross_state else ".")
        print(f"   {re.sub(r'\\s+', ' ', x.raw)[:48]:<50} "
              f"{_s(x.gnaf_label)[:40]:<42} {flags} "
              f"st{x.street_sim:.2f} loc{x.loc_sim:.2f}")

status
RESOLVED          748
STREET_ONLY       170
WEAK_MATCH         56
LOCALITY_SHIFT     33
UNRESOLVABLE        9
Name: count, dtype: int64 

status       LOCALITY_SHIFT  RESOLVED  STREET_ONLY  UNRESOLVABLE  WEAK_MATCH
change_type                                                                 
LOC                       2       199           12             0           0
LOC-STN                   7       110           11             1          13
NOF                       0        85           97             0           1
PPC                       7        38            6             0           1
PPC-STATE                 0         2            0             0           0
RETIRED                   1        26           32             0           4
STN                      15       125            8             7          33
STT                       1        36            1             1           1
UNKNOWN                   0       127            3             0           3 

RESOLV

In [ ]:
# Cell 15r: rescue pass over WEAK_MATCH and UNRESOLVABLE. Cell 12 compares
# street names as written, so an abbreviation or a phonetic spelling (MT for
# MOUNT, DICKSON for DIXON) can fall just under the 0.90 street threshold.
# This cell retries all candidates of those addresses with a looser street
# test, trusted only when the place also agrees, and promotes any that now
# qualify. Nothing is demoted.

from rapidfuzz.distance import Levenshtein

RESCUE_ABBREV = {"MT": "MOUNT", "ST": "SAINT", "PT": "PORT", "NTH": "NORTH",
                 "STH": "SOUTH", "GR": "GREAT", "LT": "LITTLE"}

def _exp(s):
    s = re.sub(r"[^\w\s'-]", " ", _s(s).upper())
    s = re.sub(r"\s+", " ", s).strip()
    return " ".join(RESCUE_ABBREV.get(t, t) for t in s.split())

def _squeeze(s):  return re.sub(r"[\s\-']", "", _s(s).upper())
def _deposs(s):
    s = re.sub(r"(?<=\w)'S\b|(?<=\w)S'\b", "", _s(s).upper())
    return re.sub(r"(?<=[A-Z]{3})S\b", "", s)

def _soundex(s):
    s = re.sub(r"[^A-Z]", "", _s(s).upper())
    if not s: return ""
    codes = {**dict.fromkeys("BFPV","1"), **dict.fromkeys("CGJKQSXZ","2"),
             **dict.fromkeys("DT","3"), "L":"4", **dict.fromkeys("MN","5"),
             "R":"6"}
    out, prev = s[0], codes.get(s[0], "")
    for ch in s[1:]:
        c = codes.get(ch, "")
        if c and c != prev: out += c
        if ch not in "HW":  prev = c
    return (out + "000")[:4]

def rescue_street_eq(a, b):
    """True when two street names are plausibly the same word."""
    ea, eb = _exp(a), _exp(b)
    if not ea or not eb:                      return False
    if ea == eb:                              return True
    if _squeeze(ea) == _squeeze(eb):          return True
    if _deposs(ea) == _deposs(eb):            return True
    d = Levenshtein.distance(ea, eb)
    if d <= 1:                                return True
    if d == 2 and min(len(ea), len(eb)) >= 5: return True
    if d <= 3 and _soundex(_squeeze(ea)) == _soundex(_squeeze(eb)):
        return True                           # DICKSON / DIXON
    return False

lost = best[best.status.isin(["WEAK_MATCH", "UNRESOLVABLE"])]
cand = resolved[resolved.raw.isin(lost.raw)].copy()
print(f"revisiting {lost.raw.nunique()} raws, {len(cand)} candidates")

cand["number_found"] = cand["number_found"].fillna(False).astype(bool)
cand["r_street"] = [rescue_street_eq(a, b)
                    for a, b in zip(cand.street_n, cand.g_street)]
cand["r_place"] = ((cand.src_postcode == cand.g_postcode)
                   & (cand.src_state == cand.g_state)) | (cand.loc_sim >= 0.90)
cand["r_tier"] = np.select(
    [cand.r_street & cand.r_place & cand.number_found,
     cand.r_street & cand.r_place],
    [1, 2], default=5)

promo = (cand[cand.r_tier < 5]
         .sort_values(["r_tier", "rank", "sim"], ascending=[True, True, False])
         .drop_duplicates("raw", keep="first"))     # whole rows, as in cell 15
promo["tier"]   = promo.r_tier
promo["status"] = promo.tier.map({1: "RESOLVED", 2: "STREET_ONLY"})
promo["type_differs"] = ((promo.type_n != promo.g_type)
                         & (promo.type_n != "") & (promo.g_type != ""))
promo["locality_differs"] = (((promo.src_postcode == promo.g_postcode)
                              & (promo.src_state == promo.g_state))
                             & (promo.loc_sim < 0.90))
promo["cross_state"] = promo.src_state != promo.g_state

rescued_raws = set(promo.raw)
print(f"\nrescued {len(rescued_raws)} addresses:")
for x in promo.itertuples():
    print(f"  {x.raw[:52]:<54} -> {x.g_street} {x.g_type} "
          f"{x.g_locality} [{x.status}]")

keep_cols = [c for c in promo.columns if c in best.columns]
best = pd.concat([best[~best.raw.isin(rescued_raws)],
                  promo[keep_cols]], ignore_index=True)
print("\nstatus after rescue:")
print(best.status.value_counts())

revisiting 65 raws, 1625 candidates

rescued 16 addresses:
  604/40-48 Aitchison Street, St Leonards, New South W   -> ATCHISON STREET ST LEONARDS [RESOLVED]
  20 Paut Street, Auburn, New South Wales, 2144          -> PAUL STREET AUBURN [RESOLVED]
  17 A Gaiwood Place, Castle Hill NSW 2154               -> GAIWOOD PLACE CASTLE HILL [RESOLVED]
  10 Mt D'Aguilar Crescent ALGESTER QLD 4115             -> MOUNT D'AGUILAR CRESCENT ALGESTER [RESOLVED]
  44 Dickson Street
    GOULBURN NSW 2580                -> DIXON STREET GOULBURN [RESOLVED]
  7 Saint Anthony Court,
                                -> ST ANTHONY COURT LARA [RESOLVED]
  Unit 12/33-35 Saint Ann Street, Merrylands NSW 2160    -> ST ANN STREET MERRYLANDS [RESOLVED]
  300 Gearga St
Sydney NSW 2002                          -> GEORGE STREET SYDNEY [RESOLVED]
  Shop 1 Morris Arcade Bowra Street Urunga NSW 2455      -> MORGO STREET URUNGA [RESOLVED]
  191 – 195 Rynie Street, Geelong, Victoria 3218         -> RYRIE STREET GEELONG [RES

In [ ]:
# Cell 16: separate street-suffix differences (FISHER RD against FISHER RD
# NORTH) from real suburb differences. The source suffix is brought back from
# to_match, both sides are put in canonical form, and a missing G-NAF suffix
# is read from the address label.

b = best.copy()

# Bring the source-side suffix back if it is missing
if "suffix_n" not in b.columns:
    b = b.merge(to_match[["raw", "suffix_n"]].drop_duplicates("raw"),
                on="raw", how="left")

for col in ["suffix_n", "g_suffix"]:
    if col not in b.columns:
        b[col] = ""
    b[col] = b[col].map(_s)

def canon_sfx(s):
    s = _s(s).upper().strip()
    return SUFFIX_MAP.get(s, s)

b["src_suffix_n"] = b.suffix_n.map(canon_sfx)
b["g_suffix_n"]   = b.g_suffix.map(canon_sfx)

# Fallback: if g_suffix is empty, look for a trailing direction in the label
SUFFIX_WORDS = {v for v in SUFFIX_MAP.values() if v}
TYPE_WORDS   = set(TYPE_MAP.values())

def suffix_from_label(label):
    toks = _s(label).upper().replace(",", " ").split()
    for i, t in enumerate(toks[:-1]):
        if t in TYPE_WORDS and toks[i + 1] in SUFFIX_WORDS:
            return toks[i + 1]
    return ""

need = b.g_suffix_n == ""
b.loc[need, "g_suffix_n"] = b.loc[need, "gnaf_label"].map(suffix_from_label)

b["suffix_differs"] = b.g_suffix_n != b.src_suffix_n
b["suffix_added"]   = (b.src_suffix_n == "") & (b.g_suffix_n != "")  # source omitted it
b["suffix_dropped"] = (b.src_suffix_n != "") & (b.g_suffix_n == "")  # source invented it

# A real suburb difference, ignoring suffix noise
b["true_loc_differs"] = (b.locality_clean.map(_s).str.strip()
                         != b.g_locality.map(_s).str.strip())

res = b[b.status == "RESOLVED"]
print(f"among RESOLVED ({len(res):,}):")
for col in ["suffix_differs", "suffix_added", "suffix_dropped",
            "true_loc_differs", "type_differs", "cross_state"]:
    print(f"  {col:20s} {int(res[col].sum()):>4,}")

print("\nsuffix cases:")
for x in res[res.suffix_differs].head(8).itertuples():
    print(f"   {re.sub(r'\\s+',' ',x.raw)[:44]:<46} {_s(x.gnaf_label)[:44]}")

print("\ngenuine suburb differences (suffix not involved):")
for x in res[res.true_loc_differs & ~res.suffix_differs].head(10).itertuples():
    print(f"   {re.sub(r'\\s+',' ',x.raw)[:44]:<46} {_s(x.gnaf_label)[:44]}")

best = b

among RESOLVED (758):
  suffix_differs         33
  suffix_added           24
  suffix_dropped          9
  true_loc_differs      352
  type_differs           61
  cross_state             2

suffix cases:
   51 Fourth Avenue Maylands WA 6051              51 FOURTH AV EAST, MAYLANDS WA 6051
   9-11 Menangle Street, Picton, NSW 2571         9-11 MENANGLE ST WEST, PICTON NSW 2571
   10/2 Rosemont Street PUNCHBOWL NSW 2196        UNIT 10 2 ROSEMONT ST NORTH, PUNCHBOWL NSW 2
   25 HALLAM AVE
          LANE COVE NSW 2066     25 HALLAM AV, LANE COVE WEST NSW 2066
   Level 5 510 Church St North Parramatta NSW 2   UNIT 27 504-516 CHURCH ST, NORTH PARRAMATTA 
   19B Welford Circuit in North Kellyville, NSW   19 WELFORD CCT, NORTH KELLYVILLE NSW 2155
   2 Queens Parade West, Newport NSW 2106         2 QUEENS PDE, NEWPORT NSW 2106
   501/40-44 Ocean Street, Bondi, New South Wal   OCEANA 40-44 OCEAN ST NORTH, BONDI NSW 2026

genuine suburb differences (suffix not involved):
   366 Ocean Beach Road,

In [ ]:
# Cell 17: classify each suburb difference as SAME, ABBREVIATION, SPACING,
# QUALIFIER (direction word missing or added), SUB_LOCALITY, TYPO,
# MISSPELLING, OVER_CAPTURED or SUBSTITUTION (a different real place), and
# write the colloquial-to-gazetted suburb lookup. pair_norm() drops a leading
# street-type word from the source only when the G-NAF name does not start
# with it, since LANE, EAST, HILLS and PARK are also street types.

from rapidfuzz.distance import Levenshtein

LOC_ABBREV = {
    "MT": "MOUNT", "MNT": "MOUNT",
    "ST": "SAINT", "PT": "PORT",
    "NTH": "NORTH", "NRTH": "NORTH",
    "STH": "SOUTH", "STHN": "SOUTH",
    "UPR": "UPPER", "LWR": "LOWER",
    "GR": "GREAT", "LT": "LITTLE",
}
QUALIFIERS = {"NORTH", "SOUTH", "EAST", "WEST", "CENTRAL",
              "UPPER", "LOWER", "HEIGHTS", "CITY"}
POSTAL_TAGS = {"BC", "DC", "MC"}      # Business/Delivery/Mail Centre
STATE_PHRASES = ["NEW SOUTH WALES", "SOUTH AUSTRALIA", "WESTERN AUSTRALIA",
                 "NORTHERN TERRITORY", "AUSTRALIAN CAPITAL TERRITORY",
                 "QUEENSLAND", "TASMANIA", "AUSTRALIA"]
TYPE_WORDS = set(TYPE_MAP.values())

def base_norm(s):
    """Punctuation, postal tags and trailing state phrases only."""
    s = re.sub(r"[^\w\s'-]", " ", _s(s).upper())
    s = re.sub(r"\s+", " ", s).strip()
    for sp in sorted(STATE_PHRASES, key=len, reverse=True):
        m = re.search(rf"^(.+?)\s+{sp}$", s)
        if m:
            s = m.group(1).strip(); break
    toks = [t for t in s.split() if t not in POSTAL_TAGS]
    return " ".join(LOC_ABBREV.get(t, t) for t in toks).strip()

def pair_norm(src, gnf):
    """Normalise as a pair. Only drop a leading token from src if it is a
       street type AND the G-NAF side does not start with the same token."""
    a, c = base_norm(src), base_norm(gnf)
    at, ct = a.split(), c.split()
    if len(at) > 1 and at[0] in TYPE_WORDS and (not ct or ct[0] != at[0]):
        at = at[1:]
    return " ".join(at), c

def squeeze(s):
    return re.sub(r"[\s\-']", "", _s(s).upper())

b = best.copy()
b["src_loc"] = b.locality_clean.map(_s).str.strip()
b["gnf_loc"] = b.g_locality.map(_s).str.strip()

pairs = [pair_norm(a, c) for a, c in zip(b.src_loc, b.gnf_loc)]
b["src_x"] = [p[0] for p in pairs]
b["gnf_x"] = [p[1] for p in pairs]

b["loc_edit"] = [Levenshtein.distance(a, c) for a, c in zip(b.src_x, b.gnf_x)]
b["loc_norm"] = b.loc_edit / b[["src_x", "gnf_x"]].map(len).max(axis=1).clip(lower=1)

def qualifier_only(src, gnf):
    a = [t for t in src.split() if t not in QUALIFIERS]
    c = [t for t in gnf.split() if t not in QUALIFIERS]
    return a == c and a != [] and src != gnf

def contains_other(src, gnf):
    a, c = src.split(), gnf.split()
    return bool(a) and bool(c) and (
        (len(a) > len(c) and a[:len(c)] == c) or
        (len(c) > len(a) and c[:len(a)] == a))

b["qual_only"]     = [qualifier_only(a, c) for a, c in zip(b.src_x, b.gnf_x)]
b["contains"]      = [contains_other(a, c) for a, c in zip(b.src_x, b.gnf_x)]
b["squeeze_same"]  = b.src_x.map(squeeze) == b.gnf_x.map(squeeze)
b["over_captured"] = b.src_loc.map(lambda s: len(_s(s).split()) > 3)

def loc_class(r):
    if r.src_loc == r.gnf_loc: return "SAME"
    if r.src_x == r.gnf_x:     return "ABBREVIATION"
    if r.squeeze_same:         return "SPACING"
    if r.qual_only:            return "QUALIFIER"
    if r.contains:             return "SUB_LOCALITY"
    if r.loc_edit <= 2:        return "TYPO"
    if r.loc_norm <= 0.25:     return "MISSPELLING"
    if r.over_captured:        return "OVER_CAPTURED"
    return "SUBSTITUTION"

b["loc_class"] = b.apply(loc_class, axis=1)

print("sanity checks:")
for a, c in [("LANE COVE", "LANE COVE WEST"), ("EASTHILLS", "EAST HILLS"),
             ("NRTH ST MARYS", "NORTH ST MARYS"),
             ("AVENUE MT OUSLEY", "MOUNT OUSLEY"),
             ("GILBERTON SOUTH AUSTRALIA", "GILBERTON")]:
    x, y = pair_norm(a, c)
    print(f"   {a!r:30} -> {c!r:22}  {x!r} / {y!r}")

res = b[b.status == "RESOLVED"]
print(f"\nlocality classes among RESOLVED ({len(res):,}):")
print(res.loc_class.value_counts().to_string(), "\n")

for cls in ["ABBREVIATION", "SPACING", "QUALIFIER", "SUB_LOCALITY",
            "TYPO", "MISSPELLING", "OVER_CAPTURED", "SUBSTITUTION"]:
    sub = res[res.loc_class == cls]
    if not len(sub):
        continue
    print(f"── {cls} ({len(sub):,})")
    for x in sub.head(6).itertuples():
        print(f"     {x.src_loc[:28]:<30} -> {x.gnf_loc[:28]:<30} (d={x.loc_edit})")
    print()

best = b

sub = best[(best.status == "RESOLVED") & (best.loc_class == "SUBSTITUTION")]
lookup = (sub.groupby(["src_loc", "gnf_loc", "g_state"])
            .size().reset_index(name="n").sort_values("n", ascending=False))
lookup.to_csv(COLLOQUIAL_CSV, index=False)
print(f"{len(lookup):,} colloquial pairs written to {COLLOQUIAL_CSV}")
print(lookup.head(20).to_string(index=False))

sanity checks:
   'LANE COVE'                    -> 'LANE COVE WEST'        'LANE COVE' / 'LANE COVE WEST'
   'EASTHILLS'                    -> 'EAST HILLS'            'EASTHILLS' / 'EAST HILLS'
   'NRTH ST MARYS'                -> 'NORTH ST MARYS'        'NORTH SAINT MARYS' / 'NORTH SAINT MARYS'
   'AVENUE MT OUSLEY'             -> 'MOUNT OUSLEY'          'MOUNT OUSLEY' / 'MOUNT OUSLEY'
   'GILBERTON SOUTH AUSTRALIA'    -> 'GILBERTON'             'GILBERTON' / 'GILBERTON'

locality classes among RESOLVED (758):
loc_class
SAME            406
SUBSTITUTION    146
QUALIFIER       112
TYPO             54
SUB_LOCALITY     15
ABBREVIATION     15
SPACING           8
MISSPELLING       2 

── ABBREVIATION (15)
     GILBERTON SOUTH AUSTRALIA      -> GILBERTON                      (d=0)
     FORTITUDE VALLEY BC            -> FORTITUDE VALLEY               (d=0)
     SOUTH CRONULLA                 -> CRONULLA                       (d=0)
     NRTH ST MARYS                  -> NORTH ST MARYS        

In [ ]:
# Cell 18: classify each street-name difference as SAME, SPACING_APOS
# (spacing or apostrophe), POSSESSIVE (GOSWELLS against GOSWELL), EDIT1 or
# EDIT2 (one or two characters), MISSPELLING (more, but the same word) or
# DIFFERENT, and cross-tabulate it against the suburb class.

from rapidfuzz.distance import Levenshtein

ST_ABBREV = {
    "MT": "MOUNT", "ST": "SAINT", "PT": "PORT",
    "NTH": "NORTH", "STH": "SOUTH", "GR": "GREAT", "LT": "LITTLE",
    "MC": "MC", "OLD": "OLD",
}
POSSESSIVE = re.compile(r"(?<=\w)'S\b|(?<=\w)S'\b")

def street_norm(s):
    s = re.sub(r"[^\w\s'-]", " ", _s(s).upper())
    s = re.sub(r"\s+", " ", s).strip()
    return " ".join(ST_ABBREV.get(t, t) for t in s.split())

def depossess(s):
    """GOSWELLS -> GOSWELL, ST GEORGE'S -> ST GEORGE"""
    s = POSSESSIVE.sub("", _s(s).upper())
    return re.sub(r"(?<=[A-Z]{3})S\b", "", s)

def squeeze(s):
    return re.sub(r"[\s\-']", "", _s(s).upper())

b = best.copy()
b["src_st"] = b.street_n.map(street_norm)
b["gnf_st"] = b.g_street.map(street_norm)

# Whether expanding abbreviations (Mt, Saint, Nth) changed the source street.
# If so and the expanded forms agree, the only difference was an abbreviation.
b["st_had_abbrev"] = [
    street_norm(s) != re.sub(r"\s+", " ", _s(s).upper()).strip()
    for s in b.street_n
]

b["st_edit"] = [Levenshtein.distance(a, c) for a, c in zip(b.src_st, b.gnf_st)]
b["st_len"]  = b[["src_st", "gnf_st"]].map(len).max(axis=1).clip(lower=1)
b["st_rel"]  = b.st_edit / b.st_len

b["st_squeeze_same"] = b.src_st.map(squeeze)  == b.gnf_st.map(squeeze)
b["st_poss_same"]    = b.src_st.map(depossess) == b.gnf_st.map(depossess)
b["st_apos"]         = (b.src_st.str.contains("'") != b.gnf_st.str.contains("'"))

def st_class(r):
    if r.src_st == r.gnf_st:   return "SAME"
    if r.st_squeeze_same:      return "SPACING_APOS"   # SALTWATER / SALT WATER
    if r.st_poss_same:         return "POSSESSIVE"     # GOSWELLS / GOSWELL
    if r.st_edit == 1:         return "EDIT1"          # single character
    if r.st_edit == 2:         return "EDIT2"
    if r.st_rel <= 0.30:       return "MISSPELLING"
    return "DIFFERENT"

b["st_class"] = b.apply(st_class, axis=1)

res = b[b.status == "RESOLVED"]
print(f"street classes among RESOLVED ({len(res):,}):")
print(res.st_class.value_counts().to_string(), "\n")

for cls in ["SPACING_APOS", "POSSESSIVE", "EDIT1", "EDIT2",
            "MISSPELLING", "DIFFERENT"]:
    sub = res[res.st_class == cls]
    if not len(sub):
        continue
    print(f"── {cls} ({len(sub):,})")
    for x in sub.head(8).itertuples():
        print(f"     {x.src_st[:26]:<28} -> {x.gnf_st[:26]:<28} (d={x.st_edit})")
    print()

# Joint view: which fields go wrong together?
print("locality class x street class (RESOLVED):")
print(pd.crosstab(res.loc_class, res.st_class))

best = b

street classes among RESOLVED (758):
st_class
SAME            602
EDIT1            78
SPACING_APOS     23
POSSESSIVE       18
DIFFERENT        18
EDIT2            16
MISSPELLING       3 

── SPACING_APOS (23)
     BEACONHILLS                  -> BEACON HILLS                 (d=1)
     SAINT GEORGE'S               -> SAINT GEORGES                (d=1)
     SAINT GEORGE'S               -> SAINT GEORGES                (d=1)
     GLENHUNTLY                   -> GLEN HUNTLY                  (d=1)
     GREENHILLS                   -> GREEN HILLS                  (d=1)
     SAINT GEORGE'S               -> SAINT GEORGES                (d=1)
     SAINT GEORGE'S               -> SAINT GEORGES                (d=1)
     SAINT GEORGE'S               -> SAINT GEORGES                (d=1)

── POSSESSIVE (18)
     DOUG MCWILLIAMS              -> DOUG MCWILLIAM               (d=1)
     PENGUIN HEAD                 -> PENGUINS HEAD                (d=1)
     SAINT GEORGE                 -> SAINT GEORGES 

In [ ]:
# Cell 19: split WORD_DROP (a whole word missing or added) out of DIFFERENT,
# count each class by row and by distinct pair, and find which field carries
# the error. The distinct count matters because the same address can appear
# in many court documents. Writes output/resolved_classified.parquet.

res = best[best.status == "RESOLVED"].copy()

def word_subset(a, c):
    """One name is the other with whole words added or removed."""
    at, ct = a.split(), c.split()
    return bool(at) and bool(ct) and (set(at) < set(ct) or set(ct) < set(at))

res["word_drop"] = [word_subset(a, c) for a, c in zip(res.src_st, res.gnf_st)]
res.loc[(res.st_class == "DIFFERENT") & res.word_drop, "st_class"] = "WORD_DROP"

print("STREET, by row vs by distinct pair:")
rows = res.st_class.value_counts()
dist = (res.drop_duplicates(["src_st", "gnf_st"]).st_class.value_counts())
print(pd.DataFrame({"rows": rows, "distinct": dist})
        .fillna(0).astype(int).to_string(), "\n")

print("LOCALITY, by row vs by distinct pair:")
rows_l = res.loc_class.value_counts()
dist_l = (res.drop_duplicates(["src_x", "gnf_x"]).loc_class.value_counts())
print(pd.DataFrame({"rows": rows_l, "distinct": dist_l})
        .fillna(0).astype(int).to_string(), "\n")

# Which single field carries the error?
def field(r):
    s = r.st_class != "SAME"
    l = r.loc_class != "SAME"
    return "BOTH" if s and l else "STREET" if s else "LOCALITY" if l else "NEITHER"

res["error_field"] = res.apply(field, axis=1)
vc = res.error_field.value_counts()
print("which field carries the error:")
print(pd.DataFrame({"n": vc, "pct": (100*vc/len(res)).round(1)}).to_string())

print("\nremaining true DIFFERENT (street):")
for x in res[res.st_class == "DIFFERENT"].itertuples():
    print(f"   {x.src_st[:26]:<28} -> {x.gnf_st[:26]}")

res.to_parquet(CLASSIFIED_PQ, index=False)
print(f"\nwrote {len(res):,} rows to {CLASSIFIED_PQ}")

best = best.merge(res[["raw", "st_class", "error_field"]].rename(
    columns={"st_class": "st_class_v2"}), on="raw", how="left")

STREET, by row vs by distinct pair:
              rows  distinct
st_class                    
DIFFERENT        8         8
EDIT1           78        73
EDIT2           16        16
MISSPELLING      3         3
POSSESSIVE      18        14
SAME           602       438
SPACING_APOS    23        14
WORD_DROP       10         8 

LOCALITY, by row vs by distinct pair:
              rows  distinct
loc_class                   
SAME           406       269
SUBSTITUTION   146        96
QUALIFIER      112        56
TYPO            54        53
SUB_LOCALITY    15        13
ABBREVIATION    15        12
SPACING          8         8
MISSPELLING      2         2 

which field carries the error:
               n   pct
error_field           
LOCALITY     323  42.6
NEITHER      279  36.8
STREET       127  16.8
BOTH          29   3.8

remaining true DIFFERENT (street):
   RIVERVIEW                    -> RIVER
   EVERLEIGH                    -> EVERLEY
   MARD                         -> MARSDEN
   CAMBERW

In [ ]:
# Cell 20: character-level edit operations for each misspelt street or
# suburb: which characters changed, where in the word (first two, middle,
# last two) and in which direction (clean to dirty, the way a correct address
# becomes misspelt). Reads the classified rows from the file cell 19 wrote.

from rapidfuzz.distance import Levenshtein
from collections import Counter

res = pd.read_parquet(CLASSIFIED_PQ)

CHAR_CLASSES = {"EDIT1", "EDIT2", "MISSPELLING"}
LOC_CHAR     = {"TYPO", "MISSPELLING"}

def ops_for(dirty, clean):
    """Edit operations that turn the clean name into the dirty one."""
    out = []
    for op in Levenshtein.editops(clean, dirty):
        pos, n = op.src_pos, len(clean)
        band = "start" if pos < 2 else "end" if pos >= n - 2 else "mid"
        if op.tag == "replace":
            out.append(("substitute", clean[op.src_pos], dirty[op.dest_pos], band))
        elif op.tag == "insert":
            out.append(("insert", "", dirty[op.dest_pos], band))
        elif op.tag == "delete":
            out.append(("delete", clean[op.src_pos], "", band))
    return out

rows = []
for x in res.itertuples():
    if x.st_class in CHAR_CLASSES and x.src_st != x.gnf_st:
        for tag, frm, to, band in ops_for(x.src_st, x.gnf_st):
            rows.append(("street", tag, frm, to, band, x.gnf_st, x.src_st))
    if x.loc_class in LOC_CHAR and x.src_x != x.gnf_x:
        for tag, frm, to, band in ops_for(x.src_x, x.gnf_x):
            rows.append(("locality", tag, frm, to, band, x.gnf_x, x.src_x))

ops = pd.DataFrame(rows, columns=["field", "op", "from", "to", "band",
                                  "clean", "dirty"])
print(f"{len(ops):,} character operations from "
      f"{ops[['clean','dirty']].drop_duplicates().shape[0]:,} distinct pairs\n")

print("operation type:")
print((ops.op.value_counts(normalize=True) * 100).round(1).to_string(), "\n")

print("position in word:")
print((ops.band.value_counts(normalize=True) * 100).round(1).to_string(), "\n")

print("top substitutions (clean -> dirty):")
subs = ops[ops.op == "substitute"]
print((subs.groupby(["from", "to"]).size()
       .sort_values(ascending=False).head(20).to_string()), "\n")

print("most-deleted characters:")
print(ops[ops.op == "delete"]["from"].value_counts().head(12).to_string(), "\n")
print("most-inserted characters:")
print(ops[ops.op == "insert"]["to"].value_counts().head(12).to_string())

# A transposition shows as two adjacent substitutions, so count it apart
def is_transpose(a, b):
    if len(a) != len(b): return False
    d = [i for i in range(len(a)) if a[i] != b[i]]
    return len(d) == 2 and d[1] == d[0] + 1 and a[d[0]] == b[d[1]] and a[d[1]] == b[d[0]]

tr = res[[is_transpose(a, c) for a, c in zip(res.src_st, res.gnf_st)]]
print(f"\ntranspositions: {len(tr)}")
for x in tr.head(8).itertuples():
    print(f"   {x.src_st} -> {x.gnf_st}")

ops.to_parquet(CHAR_OPS_PQ, index=False)
print(f"\nwrote {len(ops):,} rows to {CHAR_OPS_PQ}")

195 character operations from 144 distinct pairs

operation type:
op
insert        36.4
delete        36.4
substitute    27.2 

position in word:
band
mid      51.8
end      30.8
start    17.4 

top substitutions (clean -> dirty):
from  to
E     A     5
O     A     4
A     E     3
L     I     3
R     N     2
I     L     2
      E     2
E     I     2
K     G     2
A     U     2
C     T     1
N     M     1
Y     M     1
      I     1
V     D     1
T     L     1
      A     1
S     D     1
A     N     1
R     L     1 

most-deleted characters:
from
E    8
R    8
N    7
S    7
T    7
L    7
I    5
H    4
A    4
Y    3
G    2
D    2 

most-inserted characters:
to
S    11
R    11
E     9
A     7
      4
I     4
G     4
L     4
D     3
Y     3
N     2
U     2

transpositions: 4
   KEIWARRA -> KIEWARRA
   KRISTINA -> KIRSTINA
   ASPLEY -> APSLEY
   TYRON -> TRYON

wrote 195 rows to output/char_edit_ops.parquet


In [ ]:
# Cell 21: the misspellings TSV. One row per address: the address as written,
# the matched G-NAF address, and a reason listing every fault. Rows matched
# to retired G-NAF addresses are left out, since they were correct when
# written.

b = best.copy()

# Street class from cell 19's file, which holds RESOLVED rows only, so other
# rows have no street class.
rc = pd.read_parquet(CLASSIFIED_PQ)
b = b.drop(columns=[c for c in ["st_class", "error_field"] if c in b.columns])
b = b.merge(rc[["raw", "st_class", "error_field"]].drop_duplicates("raw"),
            on="raw", how="left")

# Drop retired matches: valid at the time of writing, not mistakes
b = b[b.change_type != "RETIRED"].copy()
print(f"{len(b):,} rows after excluding RETIRED")

def clean(s):
    return re.sub(r"\s+", " ", _s(s)).strip()

LOC_REASON = {
    "SUBSTITUTION": "Wrong suburb (a different real locality)",
    "QUALIFIER":    "Suburb missing or gaining a direction word (e.g. Queanbeyan vs Queanbeyan East)",
    "TYPO":         "Suburb misspelt (1-2 characters)",
    "SUB_LOCALITY": "Suburb is an unofficial or sub-locality name",
    "ABBREVIATION": "Suburb abbreviated (e.g. Mt for Mount)",
    "SPACING":      "Suburb word spacing wrong (e.g. Ferntree vs Fern Tree)",
    "MISSPELLING":  "Suburb misspelt (3+ characters)",
    "OVER_CAPTURED":"Extra words captured with the suburb",
}
ST_REASON = {
    "EDIT1":        "Street name misspelt (1 character)",
    "EDIT2":        "Street name misspelt (2 characters)",
    "MISSPELLING":  "Street name misspelt (3+ characters)",
    "POSSESSIVE":   "Street name plural or possessive S wrong",
    "SPACING_APOS": "Street name spacing or apostrophe wrong",
    "WORD_DROP":    "Street name missing or gaining a word",
    "DIFFERENT":    "Street name substantially different",
}

def reason(r):
    if r.status not in ("RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT"):
        return "Cannot be matched"
    parts = []
    if _s(r.get("st_class")) not in ("", "SAME", "nan"):
        parts.append(ST_REASON.get(r.st_class, r.st_class))
    if bool(r.get("st_had_abbrev")) and _s(r.get("st_class")) in ("SAME", ""):
        parts.append("Street name abbreviated (e.g. St for Saint)")
    if _s(r.get("loc_class")) not in ("", "SAME", "nan"):
        parts.append(LOC_REASON.get(r.loc_class, r.loc_class))
    if bool(r.get("type_differs")):
        parts.append("Street type wrong (e.g. Road vs Avenue)")
    if bool(r.get("suffix_differs")):
        parts.append("Street direction suffix wrong (e.g. North missing)")
    if _s(r.get("src_postcode")) != _s(r.get("g_postcode")):
        parts.append("Postcode does not match")
    if bool(r.get("cross_state")):
        parts.append("State does not match")
    # LOCALITY_SHIFT does not need a confirmed number either, so it gets the
    # same number caveat as STREET_ONLY when none was found.
    if r.status in ("STREET_ONLY", "LOCALITY_SHIFT") and not bool(r.get("number_found", False)):
        parts.append("Street number does not exist on this street")
    if not parts:
        parts.append("Unit, level or lot detail differs")
    return "; ".join(parts)

b["reason"] = b.apply(reason, axis=1)
b["matched"] = np.where(b.status.isin(["RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT"]),
                        b.gnaf_label.map(clean), "")
b.loc[(b.matched == "") & (b.status != "UNRESOLVABLE"), "matched"] = (
    b.g_street.map(clean) + " " + b.g_type.map(clean) + " "
    + b.g_locality.map(clean) + " " + b.g_state.map(clean) + " "
    + b.g_postcode.map(clean)).str.strip()

out = pd.DataFrame({
    "address":  b.raw.map(clean),
    "matched":  b.matched.map(clean),
    "reason":   b.reason,
}).sort_values(["reason", "address"]).reset_index(drop=True)

# Once misspellings.tsv carries an adjudication column it is the test set
# after the manual label review, and writing over it would erase those
# corrections. In that case the output goes to misspellings.harvest.tsv.
_dest = MISSPELLINGS_TSV
if MISSPELLINGS_TSV.exists():
    _cols = pd.read_csv(MISSPELLINGS_TSV, sep="\t", nrows=0).columns
    if "adjudication" in _cols:
        _dest = EXPORT / "misspellings.harvest.tsv"
        print("*" * 72)
        print(f"WARNING: {MISSPELLINGS_TSV} carries an adjudication column: it is the")
        print("         hand-corrected test set and must not be overwritten by the harvest.")
        print(f"         Writing this run's output to {_dest} instead.")
        print("*" * 72)
out.to_csv(_dest, sep="\t", index=False, lineterminator="\n")
print(f"wrote {len(out):,} rows to {_dest}\n")
out.head(15)

In [ ]:
# Cell 22: mutation tables. mutations_long.tsv has one row per address and
# mutation, so an address with two faults appears twice. mutation_types.tsv
# gives each mutation type as a share of all mutations and of all addresses.

rows = []
for r in b.itertuples():
    if r.status not in ("RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT"):
        rows.append((r.raw, "UNMATCHABLE", "Cannot be matched", "other"))
        continue
    sc, lc = _s(getattr(r, "st_class", "")), _s(getattr(r, "loc_class", ""))
    if sc not in ("", "SAME", "nan"):
        rows.append((r.raw, f"STREET_{sc}", ST_REASON.get(sc, sc), "street"))
    if lc not in ("", "SAME", "nan"):
        rows.append((r.raw, f"LOCALITY_{lc}", LOC_REASON.get(lc, lc), "locality"))
    if bool(getattr(r, "type_differs", False)):
        rows.append((r.raw, "STREET_TYPE", "Street type wrong", "street"))
    if bool(getattr(r, "suffix_differs", False)):
        rows.append((r.raw, "STREET_SUFFIX", "Direction suffix wrong", "street"))
    if _s(getattr(r, "src_postcode", "")) != _s(getattr(r, "g_postcode", "")):
        rows.append((r.raw, "POSTCODE", "Postcode does not match", "postcode"))
    if bool(getattr(r, "cross_state", False)):
        rows.append((r.raw, "STATE", "State does not match", "state"))
    if r.status == "STREET_ONLY":
        rows.append((r.raw, "NUMBER", "Number not on this street", "number"))

mut = pd.DataFrame(rows, columns=["raw", "code", "description", "field"])

summary = (mut.groupby(["field", "code", "description"])
             .size().reset_index(name="n")
             .sort_values("n", ascending=False))
summary["pct_of_mutations"] = (100 * summary.n / len(mut)).round(2)
summary["pct_of_addresses"] = (100 * summary.n / b.raw.nunique()).round(2)

print(f"{len(mut):,} mutations across {b.raw.nunique():,} addresses "
      f"({len(mut)/b.raw.nunique():.2f} per address)\n")
print(summary.to_string(index=False), "\n")

print("by field:")
fs = mut.groupby("field").size().sort_values(ascending=False)
print(pd.DataFrame({"n": fs, "pct": (100*fs/len(mut)).round(1)}).to_string(), "\n")

print("mutations per address:")
per = mut.groupby("raw").size().value_counts().sort_index()
print(pd.DataFrame({"addresses": per,
                    "pct": (100*per/per.sum()).round(1)}).to_string())

summary.to_csv(MUT_TYPES_TSV, sep="\t", index=False)
mut.to_csv(MUT_LONG_TSV, sep="\t", index=False)
print(f"\nwrote {MUT_TYPES_TSV.name} and {MUT_LONG_TSV.name}")

1,063 mutations across 953 addresses (1.12 per address)

   field                  code                                                                     description   n  pct_of_mutations  pct_of_addresses
postcode              POSTCODE                                                         Postcode does not match 215             20.23             22.56
locality LOCALITY_SUBSTITUTION                                        Wrong suburb (a different real locality) 178             16.75             18.68
  number                NUMBER                                                       Number not on this street 144             13.55             15.11
locality    LOCALITY_QUALIFIER Suburb missing or gaining a direction word (e.g. Queanbeyan vs Queanbeyan East) 115             10.82             12.07
  street          STREET_EDIT1                                              Street name misspelt (1 character)  78              7.34              8.18
  street           STREET_TYPE       

In [ ]:
# Cell 22b: rates of two format differences outside the reason taxonomy: an
# occupancy prefix (unit word, slash form or lot) that the matched G-NAF
# address lacks, and abbreviations (St, Rd, Ave, Mt, Nth) or a state written
# in full. Rates are over matched rows only, since an unmatchable row has no
# G-NAF address to compare with.

import re
from collections import Counter

tsv = pd.read_csv("output/misspellings.tsv", sep="\t").fillna("")
m = tsv[tsv.reason != "Cannot be matched"].copy()
M = len(m)
print(f"{M:,} matched rows form the denominator\n")

UNIT_KW = r"\b(UNIT|LEVEL|SHOP|SUITE|FLAT|APT|APARTMENT)\b"

def prefix_kind(a, g):
    a_u, g_u = a.upper(), g.upper()
    if re.search(r"\bLOT\b", a_u) and not re.search(r"\bLOT\b", g_u):
        return "lot"
    if re.search(UNIT_KW, a_u) and not re.search(UNIT_KW, g_u):
        return "unit_kw"
    if re.search(r"\d+[A-Z]?\s*/\s*\d+", a_u) and not re.search(r"/\s*\d", g_u):
        return "slash"
    return ""

m["prefix_kind"] = [prefix_kind(a, g) for a, g in zip(m.address, m.matched)]
pk = m.prefix_kind.value_counts()
n_prefix = int((m.prefix_kind != "").sum())
prefix_rate = {
    "any":     n_prefix / M,
    "unit_kw": int(pk.get("unit_kw", 0)) / M,
    "slash":   int(pk.get("slash", 0))   / M,
    "lot":     int(pk.get("lot", 0))     / M,
    "range_written_gnaf_single": int(sum(
        bool(re.search(r"\d+\s*[-\u2013]\s*\d+", a)) and
        not re.search(r"\d+\s*[-\u2013]\s*\d+", g)
        for a, g in zip(m.address, m.matched))) / M,
    "single_written_gnaf_range": int(sum(
        bool(re.search(r"\d+\s*[-\u2013]\s*\d+", g)) and
        not re.search(r"\d+\s*[-\u2013]\s*\d+", a)
        for a, g in zip(m.address, m.matched))) / M,
}
print("occupancy prefix rates:")
for k, v in prefix_rate.items():
    print(f"  {k:<28} {v:6.1%}")

TYPE_ABBR = {"ST","RD","AVE","AV","CR","CRES","PDE","HWY","BVD","BLVD","CT",
             "PL","DR","DVE","TCE","GR","LN","CCT","CL","ESP","PKWY","SQ"}
NAME_ABBR = {"MT","NTH","STH","PT"}
STATE_FULL = ("NEW SOUTH WALES","VICTORIA","QUEENSLAND","TASMANIA",
              "SOUTH AUSTRALIA","WESTERN AUSTRALIA","NORTHERN TERRITORY",
              "AUSTRALIAN CAPITAL TERRITORY")

abbr_tokens = Counter()
flags = {"type_abbr": 0, "name_abbr": 0, "saint_abbr": 0, "state_full": 0}
for a in m.address:
    up = re.sub(r"[^\w\s/]", " ", a.upper())
    toks = up.split()
    if any(t in TYPE_ABBR for t in toks):
        flags["type_abbr"] += 1
        abbr_tokens.update(t for t in toks if t in TYPE_ABBR)
    if any(t in NAME_ABBR for t in toks):
        flags["name_abbr"] += 1
        abbr_tokens.update(t for t in toks if t in NAME_ABBR)
    if re.search(r"\bSt\.?\s+[A-Z][a-z]", a):
        flags["saint_abbr"] += 1
    if any(sf in up for sf in STATE_FULL):
        flags["state_full"] += 1

abbrev_rate = {k: v / M for k, v in flags.items()}
print("\nabbreviation rates:")
for k, v in abbrev_rate.items():
    print(f"  {k:<12} {v:6.1%}")
print("\ntoken catalogue:", dict(abbr_tokens.most_common()))

908 matched rows form the denominator

occupancy prefix rates:
  any                           18.4%
  unit_kw                        8.0%
  slash                          7.6%
  lot                            2.8%
  range_written_gnaf_single      3.7%
  single_written_gnaf_range     12.0%

abbreviation rates:
  type_abbr     10.6%
  name_abbr      1.3%
  saint_abbr     4.5%
  state_full    16.7%

token catalogue: {'ST': 68, 'RD': 19, 'MT': 11, 'AVE': 10, 'NTH': 2, 'HWY': 2, 'TCE': 1}


In [ ]:
# Cell 22c: the mutation matrix, one row per address with a flag for every
# error and format difference. Error flags match exact phrases in the reason
# column. Format flags compare the written address with the matched G-NAF
# address. Rates are given per row and per distinct pair.

import re
import pandas as pd

tsv = pd.read_csv(EXPORT / "misspellings.tsv", sep="\t").fillna("")
mut = tsv.copy()
mut["is_matched"] = mut.reason != "Cannot be matched"

# Error flags: exact reason phrases, including the text in brackets
ERROR_FLAGS = {
    "err_st_edit1":     "Street name misspelt (1 character)",
    "err_st_edit2":     "Street name misspelt (2 characters)",
    "err_st_heavy":     "Street name misspelt (3+ characters)",
    "err_st_abbrev":    "Street name abbreviated",          # from reason() in cell 21
    "err_st_possessive":"Street name plural or possessive S wrong",
    "err_st_spacing":   "Street name spacing or apostrophe wrong",
    "err_st_worddrop":  "Street name missing or gaining a word",
    "err_st_different": "Street name substantially different",
    "err_st_type":      "Street type wrong",
    "err_st_suffix":    "Street direction suffix wrong",
    "err_loc_wrong":    "Wrong suburb",
    "err_loc_qualifier":"Suburb missing or gaining a direction word",
    "err_loc_typo":     "Suburb misspelt (1-2 characters)",
    "err_loc_heavy":    "Suburb misspelt (3+ characters)",
    "err_loc_sublocal": "Suburb is an unofficial or sub-locality name",
    "err_loc_abbrev":   "Suburb abbreviated",
    "err_loc_spacing":  "Suburb word spacing wrong",
    "err_postcode":     "Postcode does not match",
    "err_state":        "State does not match",
    "err_number":       "Street number does not exist on this street",
    "err_occupancy":    "Unit, level or lot detail differs",
}
for col, phrase in ERROR_FLAGS.items():
    mut[col] = mut.reason.str.contains(phrase, regex=False)

# Format flags: detected from the two strings
UNIT_KW = r"\b(UNIT|LEVEL|SHOP|SUITE|FLAT|APT|APARTMENT)\b"
TYPE_ABBR = {"ST","RD","AVE","AV","CR","CRES","PDE","HWY","BVD","BLVD","CT",
             "PL","DR","DVE","TCE","GR","LN","CCT","CL","ESP","PKWY","SQ"}
NAME_ABBR = {"MT","NTH","STH","PT"}
STATE_FULL = ("NEW SOUTH WALES","VICTORIA","QUEENSLAND","TASMANIA",
              "SOUTH AUSTRALIA","WESTERN AUSTRALIA","NORTHERN TERRITORY",
              "AUSTRALIAN CAPITAL TERRITORY")
RANGE = r"\d+\s*[-\u2013]\s*\d+"

def fmt_flags(a, g):
    au, gu = a.upper(), g.upper()
    toks = re.sub(r"[^\w\s/]", " ", au).split()
    return {
        "fmt_prefix_lot":     bool(re.search(r"\bLOT\b", au))
                              and not re.search(r"\bLOT\b", gu),
        "fmt_prefix_unit_kw": bool(re.search(UNIT_KW, au))
                              and not re.search(UNIT_KW, gu),
        "fmt_prefix_slash":   bool(re.search(r"\d+[A-Z]?\s*/\s*\d+", au))
                              and not re.search(r"/\s*\d", gu),
        "fmt_range_written":  bool(re.search(RANGE, au))
                              and not re.search(RANGE, gu),
        "fmt_single_vs_range":bool(re.search(RANGE, gu))
                              and not re.search(RANGE, au),
        "fmt_abbr_type":      any(t in TYPE_ABBR for t in toks),
        "fmt_abbr_name":      any(t in NAME_ABBR for t in toks),
        "fmt_abbr_saint":     bool(re.search(r"\bSt\.?\s+[A-Z][a-z]", a)),
        "fmt_state_full":     any(s in au for s in STATE_FULL),
        "fmt_commas":         "," in a,
        "fmt_caps_word":      bool(re.search(
            r"\b[A-Z]{4,}\b",
            re.sub(r"\b(NSW|VIC|QLD|TAS|SA|WA|NT|ACT)\b", "", a))),
    }

fm = pd.DataFrame([fmt_flags(a, g)
                   for a, g in zip(mut.address, mut.matched)])
mut = pd.concat([mut, fm], axis=1)

# Summary per row and per distinct pair
m = mut[mut.is_matched].copy()
m["pair_key"] = (m.address.str.upper().str.replace(r"\s+", " ", regex=True)
                 .str.strip() + " => " + m.matched)
dedup = m.drop_duplicates("pair_key")
FLAG_COLS = list(ERROR_FLAGS) + list(fm.columns)

rates = pd.DataFrame({
    "token_n":   m[FLAG_COLS].sum(),
    "token_pct": m[FLAG_COLS].mean().mul(100).round(1),
    "type_n":    dedup[FLAG_COLS].sum(),
    "type_pct":  dedup[FLAG_COLS].mean().mul(100).round(1),
}).sort_values("type_n", ascending=False)
print(f"matched rows: {len(m):,} tokens, {len(dedup):,} distinct pairs\n")
print(rates.to_string())

err_cols = list(ERROR_FLAGS)
dedup["n_errors"] = dedup[err_cols].sum(axis=1)
print("\nERROR faults per record (distinct pairs):")
print(dedup.n_errors.value_counts().sort_index().to_string())

co = dedup[err_cols].astype(int)
cooc = co.T @ co
print("\nERROR co-occurrence (distinct pairs, diagonal = totals):")
print(cooc.to_string())

mut.to_csv(EXPORT / "mutation_matrix.tsv", sep="\t", index=False)
rates.to_csv(EXPORT / "mutation_rates.tsv", sep="\t")
print(f"\nwrote {EXPORT/'mutation_matrix.tsv'} and {EXPORT/'mutation_rates.tsv'}")

matched rows: 908 tokens, 908 distinct pairs

                     token_n  token_pct  type_n  type_pct
fmt_commas               644       70.9     644      70.9
err_postcode             215       23.7     215      23.7
err_loc_wrong            178       19.6     178      19.6
err_number               159       17.5     159      17.5
fmt_state_full           152       16.7     152      16.7
fmt_caps_word            149       16.4     149      16.4
err_loc_qualifier        115       12.7     115      12.7
fmt_single_vs_range      109       12.0     109      12.0
fmt_abbr_type             96       10.6      96      10.6
err_occupancy             83        9.1      83       9.1
err_st_edit1              78        8.6      78       8.6
fmt_prefix_slash          75        8.3      75       8.3
fmt_prefix_unit_kw        73        8.0      73       8.0
err_st_type               73        8.0      73       8.0
err_loc_typo              56        6.2      56       6.2
fmt_abbr_saint            

In [ ]:
# Cell 23: classify each postcode difference (two adjacent digits swapped,
# one digit wrong, same region, different region, or a PO Box or large-volume
# range, which G-NAF lacks because it holds physical addresses only) and count
# wrong states. Then write output/resolved_best.parquet, now that every
# column exists.

from rapidfuzz.distance import Levenshtein

res = best[best.status.isin(["RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT"])].copy()
res["src_pc"] = res.src_postcode.map(_s).str.strip()
res["gnf_pc"] = res.g_postcode.map(_s).str.strip()
res["src_st_code"] = res.src_state.map(_s).str.strip()
res["gnf_st_code"] = res.g_state.map(_s).str.strip()

# Large-volume and PO Box ranges: valid postcodes, absent from G-NAF
LVR_RANGES = [(1000,1999),(200,299),(800,899),(900,999),
              (5800,5999),(6800,6999),(7800,7999),(8000,8999),(9000,9999)]
def is_lvr(pc):
    try: n = int(pc)
    except: return False
    return any(lo <= n <= hi for lo, hi in LVR_RANGES)

def transposed(a, c):
    """Same digits, two adjacent ones swapped: 6102 vs 6012."""
    if len(a) != len(c) or sorted(a) != sorted(c): return False
    d = [i for i in range(len(a)) if a[i] != c[i]]
    return len(d) == 2 and d[1] == d[0] + 1

def pc_class(r):
    if r.src_pc == r.gnf_pc:      return "SAME"
    if r.src_pc == "":            return "MISSING"
    if is_lvr(r.src_pc):          return "LVR_PO_BOX"
    if transposed(r.src_pc, r.gnf_pc):                    return "TRANSPOSED"
    if Levenshtein.distance(r.src_pc, r.gnf_pc) == 1:     return "ONE_DIGIT"
    if r.src_pc[:2] == r.gnf_pc[:2]:                      return "SAME_REGION"
    return "DIFFERENT_REGION"

res["pc_class"] = res.apply(pc_class, axis=1)
res["state_wrong"] = res.src_st_code != res.gnf_st_code

PC_LABEL = {
    "LVR_PO_BOX":       "PO Box / large-volume range (not a physical postcode)",
    "TRANSPOSED":       "Two adjacent digits swapped (6102 for 6012)",
    "ONE_DIGIT":        "One digit wrong (2141 for 2142)",
    "SAME_REGION":      "Different postcode, same region (first two digits match)",
    "DIFFERENT_REGION": "Postcode from a different region entirely",
    "MISSING":          "No postcode captured",
}

bad = res[res.pc_class != "SAME"]
print(f"POSTCODE errors: {len(bad):,} of {len(res):,} addresses "
      f"({100*len(bad)/len(res):.1f}%)\n")
vc = bad.pc_class.value_counts()
print(pd.DataFrame({
    "n": vc,
    "pct_of_pc_errors": (100*vc/len(bad)).round(1),
    "description": [PC_LABEL.get(i, i) for i in vc.index],
}).to_string())

print(f"\nSTATE errors: {int(res.state_wrong.sum()):,} of {len(res):,} "
      f"({100*res.state_wrong.mean():.1f}%)")
if res.state_wrong.any():
    print(res[res.state_wrong].groupby(["src_st_code", "gnf_st_code"])
            .size().reset_index(name="n").sort_values("n", ascending=False)
            .to_string(index=False))

print("\nexamples:")
for cls in vc.index:
    sub = bad[bad.pc_class == cls]
    print(f"\n── {cls} ({len(sub)})")
    for x in sub.head(3).itertuples():
        print(f"     {re.sub(r'\\s+',' ',x.raw)[:52]:<54} "
              f"{x.src_pc} -> {x.gnf_pc}")

best = best.merge(res[["raw", "pc_class", "state_wrong"]].drop_duplicates("raw"),
                  on="raw", how="left")

best.to_parquet(RESOLVED_BEST_PQ, index=False)
print(f"\nwrote {len(best):,} rows to {RESOLVED_BEST_PQ} "
      f"(the single, final write of this file)")

POSTCODE errors: 217 of 967 addresses (22.4%)

                    n  pct_of_pc_errors                                               description
pc_class                                                                                         
ONE_DIGIT         115              53.0                           One digit wrong (2141 for 2142)
DIFFERENT_REGION   47              21.7                 Postcode from a different region entirely
SAME_REGION        26              12.0  Different postcode, same region (first two digits match)
TRANSPOSED         18               8.3               Two adjacent digits swapped (6102 for 6012)
LVR_PO_BOX         11               5.1     PO Box / large-volume range (not a physical postcode)

STATE errors: 4 of 967 (0.4%)
src_st_code gnf_st_code  n
        NSW         VIC  2
        ACT         NSW  1
        VIC         NSW  1

examples:

── ONE_DIGIT (115)
     80 William Street, Sydney NSW 2000                     2000 -> 2060
     81-95 Anzac Parade,

In [ ]:
# Cell 24: mistake counts on two bases. Strict: RESOLVED rows only. Wide: all
# matched rows, where rows marked * are inflated by the tier definitions
# (LOCALITY_SHIFT means a wrong suburb, STREET_ONLY a number not found).
# On the wide base, STREET_ONLY and LOCALITY_SHIFT rows take their street
# class from cell 18, while the misspellings TSV (cell 21) leaves it blank.

b = best.copy()

# restore the WORD_DROP split lost in the merge
if "st_class_v2" in b.columns:
    b["st_class"] = b["st_class_v2"].fillna(b["st_class"])
else:
    def word_subset(a, c):
        at, ct = _s(a).split(), _s(c).split()
        return bool(at) and bool(ct) and (set(at) < set(ct) or set(ct) < set(at))
    wd = [word_subset(a, c) for a, c in zip(b.src_st, b.gnf_st)]
    b.loc[(b.st_class == "DIFFERENT") & pd.Series(wd, index=b.index), "st_class"] = "WORD_DROP"

SPECS = [
    ("Suburb", "loc_class", "SUBSTITUTION", "Wrong suburb (a different real locality)", True),
    ("Suburb", "loc_class", "QUALIFIER",    "Direction word missing or added", True),
    ("Suburb", "loc_class", "TYPO",         "Misspelt, 1-2 characters", False),
    ("Suburb", "loc_class", "ABBREVIATION", "Abbreviated (Mt for Mount)", False),
    ("Suburb", "loc_class", "SUB_LOCALITY", "Unofficial or sub-locality name", False),
    ("Suburb", "loc_class", "SPACING",      "Word spacing wrong", False),
    ("Suburb", "loc_class", "MISSPELLING",  "Misspelt, 3+ characters", False),
    ("Street name", "st_class", "EDIT1",        "Misspelt, 1 character", False),
    ("Street name", "st_class", "EDIT2",        "Misspelt, 2 characters", False),
    ("Street name", "st_class", "SPACING_APOS", "Spacing or apostrophe wrong", False),
    ("Street name", "st_class", "POSSESSIVE",   "Plural or possessive S wrong", False),
    ("Street name", "st_class", "WORD_DROP",    "Word missing or added", False),
    ("Street name", "st_class", "MISSPELLING",  "Misspelt, 3+ characters", False),
    ("Street name", "st_class", "DIFFERENT",    "Substantially different", False),
    ("Postcode", "pc_class", "ONE_DIGIT",        "One digit wrong", False),
    ("Postcode", "pc_class", "DIFFERENT_REGION", "Different region entirely", False),
    ("Postcode", "pc_class", "SAME_REGION",      "Same region, wrong code", False),
    ("Postcode", "pc_class", "TRANSPOSED",       "Two adjacent digits swapped", False),
    ("Postcode", "pc_class", "LVR_PO_BOX",       "PO Box / large-volume range", False),
]
FLAGS = [
    ("Street type",      "type_differs",   "Wrong type (Road for Street)", False),
    ("Direction suffix", "suffix_added",   "Suffix omitted", False),
    ("Direction suffix", "suffix_dropped", "Suffix added that does not exist", False),
    ("State",            "state_wrong",    "Wrong state", False),
]

def counts(df):
    d = {}
    for grp, col, cls, lab, circ in SPECS:
        d[(grp, lab, circ)] = int((df[col] == cls).sum()) if col in df else 0
    for grp, col, lab, circ in FLAGS:
        d[(grp, lab, circ)] = int(df[col].fillna(False).sum()) if col in df else 0
    d[("Number", "Number not on that street", True)] = int((df.status == "STREET_ONLY").sum())
    return d

strict = b[b.status == "RESOLVED"]
wide   = b[b.status.isin(["RESOLVED", "STREET_ONLY", "LOCALITY_SHIFT"])]
cs, cw = counts(strict), counts(wide)

rows = [(g, l, "*" if c else "", cs[(g, l, c)], round(100*cs[(g,l,c)]/len(strict),1),
         cw[(g, l, c)], round(100*cw[(g,l,c)]/len(wide),1))
        for (g, l, c) in cs]
tbl = pd.DataFrame(rows, columns=["group", "description", "circular",
                                  "n_strict", "pct_strict", "n_wide", "pct_wide"])
tbl = tbl[tbl.n_wide > 0].sort_values(["group", "n_wide"], ascending=[True, False])

print(f"strict base (RESOLVED only)     {len(strict):,}")
print(f"wide base (all matched)         {len(wide):,}")
print("* = inflated by definition on the wide base\n")
print(tbl.to_string(index=False))
tbl.to_csv(MISTAKES_TSV, sep="\t", index=False)
print(f"\nwrote {MISTAKES_TSV}")

strict base (RESOLVED only)     758
wide base (all matched)         967
* = inflated by definition on the wide base

           group                              description circular  n_strict  pct_strict  n_wide  pct_wide
Direction suffix                           Suffix omitted                 24         3.2      26       2.7
Direction suffix         Suffix added that does not exist                  9         1.2      13       1.3
          Number                Number not on that street        *         0         0.0     176      18.2
        Postcode                          One digit wrong                104        13.7     115      11.9
        Postcode                Different region entirely                 23         3.0      47       4.9
        Postcode                  Same region, wrong code                 15         2.0      26       2.7
        Postcode              Two adjacent digits swapped                 17         2.2      18       1.9
        Postcode           

In [ ]:
# Cell 25: the character substitution table, each pair put in a category
# (vowel for vowel, visual or OCR confusion, phonetic pair, other), plus
# counts of inserted and deleted characters. The sample is small, so the
# overall shape is more reliable than any single pair.

ops = pd.read_parquet(CHAR_OPS_PQ)
subs = ops[ops.op == "substitute"].copy()

VOWELS = set("AEIOU")
# Pairs that look alike in scanned text
VISUAL = {frozenset("LI"), frozenset("O0"), frozenset("RN"), frozenset("YM"),
          frozenset("SD"), frozenset("CE"), frozenset("UV"), frozenset("B8")}
# Voiced / unvoiced and other phonetic near-pairs
PHONETIC = {frozenset("KG"), frozenset("CK"), frozenset("CT"), frozenset("QC"),
            frozenset("PB"), frozenset("FV"), frozenset("SZ"), frozenset("TD")}

def category(f, t):
    if f == "" or t == "":              return "spacing"
    if f in VOWELS and t in VOWELS:     return "vowel for vowel"
    if frozenset(f + t) in VISUAL:      return "visual / OCR"
    if frozenset(f + t) in PHONETIC:    return "phonetic"
    if f not in VOWELS and t not in VOWELS: return "consonant, other"
    return "other"

tbl = (subs.groupby(["from", "to"]).size()
       .reset_index(name="n").sort_values("n", ascending=False))
tbl["category"] = [category(f, t) for f, t in zip(tbl["from"], tbl["to"])]
tbl["from"] = tbl["from"].replace("", "(space)")
tbl["to"]   = tbl["to"].replace("", "(space)")
tbl["pct"]  = (100 * tbl.n / tbl.n.sum()).round(1)

print(f"{int(tbl.n.sum())} substitutions across {len(tbl)} distinct pairs\n")
print(tbl[["from", "to", "n", "pct", "category"]].to_string(index=False), "\n")

print("by category:")
cat = tbl.groupby("category").n.sum().sort_values(ascending=False)
print(pd.DataFrame({"n": cat, "pct": (100*cat/cat.sum()).round(1)}).to_string())

tbl.to_csv(SUBSTITUTIONS_TSV, sep="\t", index=False)

# Same treatment for insertions and deletions, which outnumber substitutions
print("\nmost inserted:")
print(ops.loc[ops.op == "insert", "to"].value_counts().head(15).to_string())
print("\nmost deleted:")
print(ops.loc[ops.op == "delete", "from"].value_counts().head(15).to_string())

53 substitutions across 36 distinct pairs

from to  n  pct         category
   E  A  5  9.4  vowel for vowel
   O  A  4  7.5  vowel for vowel
   A  E  3  5.7  vowel for vowel
   L  I  3  5.7     visual / OCR
   R  N  2  3.8     visual / OCR
   I  L  2  3.8     visual / OCR
   I  E  2  3.8  vowel for vowel
   E  I  2  3.8  vowel for vowel
   K  G  2  3.8         phonetic
   A  U  2  3.8  vowel for vowel
   C  T  1  1.9         phonetic
   N  M  1  1.9 consonant, other
   Y  M  1  1.9     visual / OCR
   Y  I  1  1.9            other
   V  D  1  1.9 consonant, other
   T  L  1  1.9 consonant, other
   T  A  1  1.9            other
   S  D  1  1.9     visual / OCR
   A  N  1  1.9            other
   R  L  1  1.9 consonant, other
   Q  C  1  1.9         phonetic
   O  H  1  1.9            other
   A  O  1  1.9  vowel for vowel
   L  T  1  1.9 consonant, other
   E  O  1  1.9  vowel for vowel
   A  R  1  1.9            other
   L  F  1  1.9 consonant, other
   K  L  1  1.9 consonant, other


In [ ]:
# Cell 26: write output/README.md, listing each file in output/ with its row
# count, the cell that made it and what it holds. Any other file is listed as
# not produced by this notebook. Run it last, since it reads what is on disk.

from datetime import date

# name -> (producing cell, deliverable or intermediate, description)
KNOWN = {
    ALL_CANDIDATES_PQ.name: ("cell 14", "deliverable",
        "All 25 candidate G-NAF streets per unmatched address, with scores and "
        "any confirmed house number."),
    COLLOQUIAL_CSV.name: ("cell 17", "deliverable",
        "Colloquial suburb name to gazetted G-NAF suburb, with state and count. "
        "A reusable lookup G-NAF itself does not publish."),
    CLASSIFIED_PQ.name: ("cell 19", "deliverable",
        "The RESOLVED addresses with final street and locality classes, "
        "WORD_DROP split applied, and which field carries the error."),
    CHAR_OPS_PQ.name: ("cell 20", "deliverable",
        "One row per character-level edit (insert, delete, substitute) with "
        "position band and direction clean to dirty."),
    MISSPELLINGS_TSV.name: ("cell 21", "deliverable",
        "One row per address: original text, matched G-NAF address, and a "
        "plain-English reason listing every fault. RETIRED matches excluded."),
    MUT_TYPES_TSV.name: ("cell 22", "deliverable",
        "Mutation-type summary with counts and percentages of mutations and "
        "of addresses."),
    MUT_LONG_TSV.name: ("cell 22", "deliverable",
        "One row per (address, mutation); an address with two faults appears "
        "twice."),
    RESOLVED_BEST_PQ.name: ("cell 23", "deliverable",
        "One best G-NAF match per address with every classification column: "
        "tier, status, street and locality classes, suffix flags, postcode "
        "class, state flag."),
    MISTAKES_TSV.name: ("cell 24", "deliverable",
        "Mistake counts on the strict base (RESOLVED rows) and wide base (all "
        "matched) side by side. Rows marked * are inflated by definition on "
        "the wide base."),
    SUBSTITUTIONS_TSV.name: ("cell 25", "deliverable",
        "Character substitutions with vowel/visual/phonetic categories, from "
        "a small sample, so the overall shape is more reliable than any one pair."),
    README_MD.name: ("cell 26", "intermediate", "This file."),
}

def row_count(p):
    try:
        if p.suffix == ".parquet":
            return f"{len(pd.read_parquet(p)):,}"
        if p.suffix in (".tsv", ".csv"):
            with p.open() as f:
                return f"{sum(1 for _ in f) - 1:,}"   # minus the header
    except Exception as e:
        return f"unreadable ({e.__class__.__name__})"
    return ""

lines = [
    "# Output folder guide",
    "",
    f"Produced by the address-harvest notebook on {date.today().isoformat()}.",
    "",
    "Real Australian addresses harvested from the Open Australian Legal Corpus "
    "that fail to match G-NAF (May 2026 release), matched back to their correct "
    "address by exhaustive fuzzy matching, and classified by mistake type. "
    "Rows matched to RETIRED G-NAF addresses are left out of misspellings.tsv.",
    "",
    "| file | rows | produced by | status | contents |",
    "|---|---|---|---|---|",
]

stale = []
for p in sorted(EXPORT.iterdir()):
    if p.name.startswith("."):
        continue
    if p.name in KNOWN:
        cell, status, desc = KNOWN[p.name]
        lines.append(f"| {p.name} | {row_count(p)} | {cell} | {status} | {desc} |")
    else:
        stale.append(p.name)

if stale:
    lines += ["", "## Not produced by this notebook", "",
              "Files in output/ that this notebook does not write:", ""]
    lines += [f"- {n}" for n in stale]

lines += [
    "",
    "## Licensing",
    "",
    "G-NAF: CC BY 4.0 with a mail-verification restriction. "
    "Open Australian Legal Corpus: open, mostly commercial-use permitted. "
    "Check both before redistribution.",
    "",
    "## Known caveats",
    "",
    "- The corpus is skewed towards city centres and litigated addresses, so "
    "mutation types should generalise but rates may not.",
    "- The character-level sample is small.",
    "- Counts by distinct pair (cell 19) are less skewed than counts by row, "
    "because addresses in frequently litigated cases repeat.",
]

README_MD.write_text("\n".join(lines) + "\n")
print(f"wrote {README_MD}")
print("\n".join(lines[:20]))
print("...")

wrote output/README.md
# Output folder guide

Produced by the address-harvest notebook on 2026-08-25.

Real Australian addresses harvested from the Open Australian Legal Corpus that fail to match G-NAF (May 2026 release), matched back to their correct address by exhaustive fuzzy matching, and classified by mistake type. Rows matched to RETIRED G-NAF addresses are left out of misspellings.tsv.

| file | rows | produced by | status | contents |
|---|---|---|---|---|
| README.md |  | cell 26 | intermediate | This file. |
| char_edit_ops.parquet | 195 | cell 20 | deliverable | One row per character-level edit (insert, delete, substitute) with position band and direction clean to dirty. |
| colloquial_locality_map.csv | 97 | cell 17 | deliverable | Colloquial suburb name to gazetted G-NAF suburb, with state and count. A reusable lookup G-NAF itself does not publish. |
| misspellings.tsv | 953 | cell 21 | deliverable | One row per address: original text, matched G-NAF address, and a plain-En